# Setting

In [10]:
#import
from torch import nn,optim
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import numpy as np
from matplotlib import pyplot as plt
import os
import copy
import math
import sys
import importlib
from tqdm.auto import tqdm

In [16]:
# 코랩 환경인지 확인하는 조건문
if 'google.colab' in sys.modules:
    print("현재 환경: Google Colab")
    # 코랩 전용 설정 (예: 드라이브 마운트)
    from google.colab import drive
    drive.mount('/content/drive')
    path='/content/drive/MyDrive/02_학업/02_연구 및 프로젝트/2512-2602_Dash 연구인턴/pytorch_practice'
    # path='/content/drive/MyDrive//pytorch_practice'
    sys.path.append(path)
else:
    print("현재 환경: Local Jupyter")
    # 로컬 전용 설정 (예: 경로 설정)
    # path = './'
    path = r'g:\내 드라이브\02_Education\02_Research_and_Projects\2512_2602_Dash Research Intern\pytorch_practice'
# from my_module import *
import my_module3 as mm
print(f"작업 경로: {path}")

현재 환경: Local Jupyter
작업 경로: g:\내 드라이브\02_Education\02_Research_and_Projects\2512_2602_Dash Research Intern\pytorch_practice


In [12]:
# importlib.reload(mm)
# import my_module2 as mm

In [17]:
model_dir = os.path.join(path, 'download')
print(os.listdir(model_dir))

root=os.path.join(path, 'data','test')
os.makedirs(root, exist_ok=True)
print(os.listdir(root))

checkpoint_dir = os.path.join(path, 'checkpoints')
print(os.listdir(checkpoint_dir))

output_dir = './saved_results'
output_dir=os.path.join(checkpoint_dir,output_dir)
print(os.listdir(output_dir))

DEVICE= 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'current device: {DEVICE}')

['cifar10_vgg16_bn-6ee7ea24.pt']
['cifar-10-batches-py', 'cifar-10-python.tar.gz']
['best_model.pt', 'CNN_CIFAR10_final_weights.pth', 'ckpt_ep1.pt', 'ckpt_ep2.pt', 'ckpt_ep3.pt', 'ckpt_ep5.pt', 'ckpt_ep10.pt', 'ckpt_ep15.pt', 'ckpt_ep20.pt', 'saved_results', 'pruned_model.pth']
['psa_results.pkl', 'psa_results.json', 'psa_results_torch.pt', 'pruning_real_state.pth', 'psa_nm_results_torch.pt', 'psa_nm(34)_re_results_torch.pt', 'psa_nm_re_results_torch.pt', 'psa_re_results_torch.pt', 'psa_vector_re_results_torch.pt', 'psa_kernel_re_results_torch.pt', 'psa_channel_re_results_torch.pt', 'psa_scaling_re_results_torch.pt']
current device: cpu


In [18]:
model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_vgg16_bn", pretrained=True).to(DEVICE)

Using cache found in C:\Users\darwin5991/.cache\torch\hub\chenyaofo_pytorch-cifar-models_master


In [19]:
BATCH_SIZE=128
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

full_train_DS = datasets.CIFAR10(root=root, train=True, download=True, transform=transform_train)
test_DS = datasets.CIFAR10(root=root, train=False, download=True, transform=transform_test)

train_size = 45000
val_size = 5000
train_DS, val_DS = random_split(full_train_DS, [train_size, val_size])

train_DL=torch.utils.data.DataLoader(train_DS, batch_size=BATCH_SIZE, shuffle=True)
val_DL = torch.utils.data.DataLoader(val_DS, batch_size=BATCH_SIZE, shuffle=False)
test_DL=torch.utils.data.DataLoader(test_DS, batch_size=BATCH_SIZE, shuffle=False)

print(f"Data loaded: Train({len(train_DS)}), Val({len(val_DS)}), Test({len(test_DS)})")

Files already downloaded and verified
Files already downloaded and verified
Data loaded: Train(45000), Val(5000), Test(10000)


In [20]:
rcorrect,org_acc=mm.Test(model, test_DL, DEVICE)
print(f"Test accuracy: {rcorrect}/{len(test_DL.dataset)} ({org_acc} %)")

Test accuracy: 9416/10000 (94.2 %)


# Pruning

In [7]:
file_path_torch = os.path.join(output_dir, 'psa_re_results_torch.pt')

# 저장된 'results' 딕셔너리 불러오기
with open(file_path_torch, 'rb') as f:
    results = torch.load(f)
print(f"Results loaded from {file_path_torch}")

# 불러온 데이터 확인 (일부)
print("Loaded results (first layer): ", results[list(results.keys())[0]])

Results loaded from g:\내 드라이브\02_학업\02_연구 및 프로젝트\2512-2602_Dash 연구인턴\pytorch_practice\checkpoints\./saved_results\psa_re_results_torch.pt
Loaded results (first layer):  [{'ratio': 0.0, 'acc': 94.2}, {'ratio': 0.05, 'acc': 94.2}, {'ratio': 0.1, 'acc': 94.1}, {'ratio': 0.15, 'acc': 94.1}, {'ratio': 0.2, 'acc': 94.1}, {'ratio': 0.25, 'acc': 94.2}, {'ratio': 0.3, 'acc': 94.1}, {'ratio': 0.35, 'acc': 94.1}, {'ratio': 0.4, 'acc': 94.1}, {'ratio': 0.45, 'acc': 94.0}, {'ratio': 0.5, 'acc': 93.9}, {'ratio': 0.55, 'acc': 93.8}, {'ratio': 0.6, 'acc': 93.8}, {'ratio': 0.65, 'acc': 93.3}, {'ratio': 0.7, 'acc': 92.5}, {'ratio': 0.75, 'acc': 90.2}, {'ratio': 0.8, 'acc': 89.2}, {'ratio': 0.85, 'acc': 88.6}, {'ratio': 0.9, 'acc': 82.1}, {'ratio': 0.95, 'acc': 65.9}]


C:\Temp\ipykernel_4272\2870564656.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  results = torch.load(f)


In [9]:
print(results.keys())

dict_keys(['features.0.weight', 'features.3.weight', 'features.7.weight', 'features.10.weight', 'features.14.weight', 'features.17.weight', 'features.20.weight', 'features.24.weight', 'features.27.weight', 'features.30.weight', 'features.34.weight', 'features.37.weight', 'features.40.weight', 'classifier.0.weight', 'classifier.3.weight', 'classifier.6.weight'])


In [10]:
ratios = {}
threshold_percent=10.0
min_acceptable_acc = org_acc - threshold_percent

for layer_name, data_list in results.items():
    if 'classifier' in layer_name:
        best_ratio = 0.0
        acc_at_best = org_acc
        for data in data_list:
            ratio = data['ratio']
            acc = data['acc']
            if acc >= min_acceptable_acc:
                best_ratio = ratio
                acc_at_best = acc
            else:
                break
            ratios[layer_name] = best_ratio
        print(f"{layer_name:<40} | {best_ratio:.2f}       | {acc_at_best:.2f}%")



classifier.0.weight                      | 0.95       | 94.00%
classifier.3.weight                      | 0.95       | 94.20%
classifier.6.weight                      | 0.80       | 92.50%


In [11]:
print(results['classifier.6.weight'][15])
print(results['classifier.6.weight'][16])
print(results['classifier.6.weight'][17])

{'ratio': 0.75, 'acc': 93.3}
{'ratio': 0.8, 'acc': 92.5}
{'ratio': 0.85, 'acc': 75.8}


In [12]:
weight_param_list = [p for p in model.named_parameters() if 'weight' in p[0] and p[1].dim() ==2]
masks = {}

pruned_model = copy.deepcopy(model)
params_dict = dict(pruned_model.named_parameters())

for layer_name, weight_param in weight_param_list:
    weight_tensor=weight_param.data.clone()
    l1_mag=torch.abs(weight_tensor)
    _, idx= torch.sort(l1_mag.flatten())

    ratio=ratios[layer_name]
    num_prune=int(weight_tensor.numel()*ratio)

    pruning_mask_flat=torch.ones_like(l1_mag.flatten(), dtype=torch.float32)
    pruning_mask_flat[idx[:num_prune]]=0
    pruning_mask=pruning_mask_flat.view_as(weight_tensor)
    masks[layer_name] = pruning_mask.to(DEVICE)

    with torch.no_grad():
        pruned_weight=weight_tensor*pruning_mask
        params_dict[layer_name].copy_(pruned_weight)



In [13]:
for layer_name, weight_param in weight_param_list:
    print(layer_name)
    print(ratios[layer_name])
print(ratios)

classifier.0.weight
0.95
classifier.3.weight
0.95
classifier.6.weight
0.8
{'classifier.0.weight': 0.95, 'classifier.3.weight': 0.95, 'classifier.6.weight': 0.8}


In [14]:
pruned_rcorrect,pruned_acc=mm.Test(pruned_model, test_DL, DEVICE)
print(f"Test accuracy: {pruned_rcorrect}/{len(test_DL.dataset)} ({pruned_acc} %)")

Test accuracy: 6829/10000 (68.3 %)


In [ ]:
# criterion = nn.CrossEntropyLoss()
# optimizer=optim.SGD(pruned_model.parameters(), lr=0.001, momentum=0.9, weight_decay=5e-4)
# EPOCH=10

# for epoch in range(EPOCH):
#         pruned_model.train()
#         running_loss = 0.0
#         for inputs, labels in train_DL:
#             inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

#             optimizer.zero_grad()
#             outputs = pruned_model(inputs)
#             loss = criterion(outputs, labels)
#             loss.backward()

#             with torch.no_grad():
#                 for name, param in pruned_model.named_parameters():
#                     if name in masks:
#                         param.grad.mul_(masks[name])
#             optimizer.step()
#             running_loss += loss.item()

#         avg_loss = running_loss / len(train_DL)
#         _, ep_acc = mm.Test(pruned_model, test_DL, DEVICE)

#         print(f"Epoch [{epoch+1}/{EPOCH}] | Loss: {avg_loss:.4f} | Test Acc: {ep_acc:.2f}%")
# finetuned_rcorrect,finetuned_acc=mm.Test(pruned_model, test_DL, DEVICE)
# print(f"Final Test accuracy after fine-tuning: {finetuned_rcorrect}/{len(test_DL.dataset)} ({finetuned_acc} %)")

KeyboardInterrupt: 

In [ ]:
# # pruned_model 저장
# save_path = os.path.join(checkpoint_dir, 'pruned_model.pth')
# torch.save(pruned_model, save_path)
# print(f"Pruned model saved to {save_path}")

Pruned model saved to /content/drive/MyDrive/02_학업/02_연구 및 프로젝트/2512-2602_Dash 연구인턴/pytorch_practice/checkpoints/pruned_model.pth


# pruning 된 model 불러오기

In [21]:
save_path = os.path.join(checkpoint_dir, 'pruned_model.pth')
pruned_model = torch.load(save_path,weights_only=False,map_location=DEVICE)
pruned_model.to(DEVICE)
finetuned_rcorrect,finetuned_acc=mm.Test(pruned_model, test_DL, DEVICE)
print(f"Final Test accuracy after fine-tuning: {finetuned_rcorrect}/{len(test_DL.dataset)} ({finetuned_acc} %)")

Final Test accuracy after fine-tuning: 9405/10000 (94.0 %)


# Quantization

In [22]:
input_sub=copy.deepcopy(torch.nn.Sequential(*list(pruned_model.features.children()),
                                            torch.nn.Flatten(start_dim=1)).to(DEVICE))
sub_layers = []
for i in range(3):
    sub_layers.append(copy.deepcopy(list(pruned_model.classifier.children())[3*i]).to(DEVICE))
num_layers = len(sub_layers)

In [23]:
bit_width = list(range(2, 20))
q_min = [ -2**(b-1) for b in bit_width ]
q_max = [  2**(b-1) - 1   for b in bit_width ]

In [24]:
bit_num=8
idx=bit_num-2
q_max_=q_max[idx]
q_min_=q_min[idx]

In [25]:
layer_params =[]
r_in_min = [torch.tensor(float('inf')).to(DEVICE) for _ in range(num_layers)]
r_in_max = [torch.tensor(float('-inf')).to(DEVICE) for _ in range(num_layers)]
r_out_min = [torch.tensor(float('inf')).to(DEVICE) for _ in range(num_layers)]
r_out_max = [torch.tensor(float('-inf')).to(DEVICE) for _ in range(num_layers)]

with torch.no_grad():
    for images, _ in val_DL:
        curr_in = input_sub(images.to(DEVICE))
        for i in range(num_layers):
            curr_out = sub_layers[i](curr_in)
            if i!=2:
                curr_out = nn.ReLU()(curr_out)
            r_in_min[i] = torch.min(r_in_min[i], torch.min(curr_in))
            r_in_max[i] = torch.max(r_in_max[i], torch.max(curr_in))
            r_out_min[i] = torch.min(r_out_min[i], torch.min(curr_out))
            r_out_max[i] = torch.max(r_out_max[i], torch.max(curr_out))
            curr_in = curr_out
    for i in range(num_layers):
        S_in = (r_in_max[i] - r_in_min[i]) / (q_max_ - q_min_)
        Z_in = torch.round(q_min_ - r_in_min[i] / S_in)

        S_out = (r_out_max[i] - r_out_min[i]) / (q_max_ - q_min_)
        Z_out = torch.round(q_min_ - r_out_min[i] / S_out)

        weight_tensor = sub_layers[i].weight.data
        bias_tensor= sub_layers[i].bias.data

        r_w_max= torch.max(torch.abs(weight_tensor))
        S_w= r_w_max / (q_max_+1)
        S_b= S_in * S_w

        q_w = torch.round(weight_tensor / S_w).clamp(-q_max_-1, q_max_)
        q_b = torch.round(bias_tensor / S_b)

        q_bias= q_b - Z_in * q_w.sum(dim=1)

        scale_factor=S_w*S_in / S_out
        log2_scale=torch.log2(scale_factor)
        rounded_log2_scale = torch.round(log2_scale)
        mag_scale=scale_factor/(2**(-16))
        print(f"Layer {i}: mag_scale={mag_scale}, rounded_log2_scale={rounded_log2_scale}")
        print(f"scale_factor={scale_factor}")
        mag_scale=torch.round(mag_scale)

        layer_params.append({
            'S_in': S_in, 'Z_in': Z_in,
            'S_out': S_out, 'Z_out': Z_out,
            'S_w': S_w,
            'S_b': S_b, 'q_w': q_w, 'q_b': q_b,
            'q_bias': q_bias,
            # 'scale_factor': scale_factor,
            # 'log2_scale': log2_scale,
            # 'rounded_log2_scale': rounded_log2_scale,
            # 'Scale_factor': 2**rounded_log2_scale
            'Scale_factor': mag_scale
        })

Layer 0: mag_scale=123.79712677001953, rounded_log2_scale=-9.0
scale_factor=0.0018889942439273
Layer 1: mag_scale=51.57672119140625, rounded_log2_scale=-10.0
scale_factor=0.0007869983091950417
Layer 2: mag_scale=21.050901412963867, rounded_log2_scale=-12.0
scale_factor=0.00032121126423589885


In [26]:
for i in range(len(layer_params)):
    print(f"------------Layer {i+1}------------")
    for key, value in layer_params[i].items():
        if key == 'q_w' or  key == 'q_b' or  key == 'q_bias':
            print(f"{key}: {value.shape}")
        else:
            print(f"{key}: {value}")

------------Layer 1------------
S_in: 0.020816311240196228
Z_in: -128.0
S_out: 0.011128438636660576
Z_out: -128.0
S_w: 0.0010098598431795835
S_b: 2.102155667671468e-05
q_w: torch.Size([512, 512])
q_b: torch.Size([512])
q_bias: torch.Size([512])
Scale_factor: 124.0
------------Layer 2------------
S_in: 0.011128438636660576
Z_in: -128.0
S_out: 0.011308320797979832
Z_out: -128.0
S_w: 0.000799719535280019
S_b: 8.899629392544739e-06
q_w: torch.Size([512, 512])
q_b: torch.Size([512])
q_bias: torch.Size([512])
Scale_factor: 52.0
------------Layer 3------------
S_in: 0.011308320797979832
Z_in: -128.0
S_out: 0.07021957635879517
Z_out: 17.0
S_w: 0.0019945772364735603
S_b: 2.255531944683753e-05
q_w: torch.Size([10, 512])
q_b: torch.Size([10])
q_bias: torch.Size([10])
Scale_factor: 21.0


In [27]:
quanted_sub_layers=[]
for i in range(num_layers):
    quanted_sub_layer = copy.deepcopy(sub_layers[i])
    with torch.no_grad():
        quanted_sub_layer.weight.copy_(layer_params[i]['q_w'])
        quanted_sub_layer.bias.copy_(layer_params[i]['q_bias'])
    quanted_sub_layers.append(quanted_sub_layer)

In [28]:
input_sub.eval()
with torch.no_grad():
    rcorrect = 0
    for images, labels in test_DL:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        curr_in = input_sub(images)
        final_out=None
        curr_in=torch.round(curr_in / layer_params[0]['S_in']) + layer_params[0]['Z_in']
        for i in range(num_layers):
            curr_out = quanted_sub_layers[i](curr_in)
            if i!=2:
                curr_out = nn.ReLU()(curr_out)
            curr_out= torch.round(curr_out * layer_params[i]['Scale_factor']*(2**(-16))) + layer_params[i]['Z_out']
            curr_in = curr_out
            final_out=(curr_out-layer_params[i]['Z_out'])*layer_params[i]['S_out']
        pred = curr_in.argmax(dim=1)
        corrects = torch.sum(pred == labels).item()
        rcorrect += corrects
    accuracy_e = rcorrect/len(test_DL.dataset)*100
    print(f"Test accuracy 1: {rcorrect}/{len(test_DL.dataset)} ({accuracy_e:.1f} %)")

Test accuracy 1: 9406/10000 (94.1 %)


In [29]:
print(layer_params[0]['q_w'].shape)
# print((layer_params[0]['q_w'][1:2,:128]))

print(sub_layers[0].weight.data.shape)
print(sub_layers[0].weight.data[1:2,:128])

torch.Size([512, 512])
torch.Size([512, 512])
tensor([[-0., 0., -0., -0., 0., -0., 0., 0., 0., -0., 0., 0., -0., 0., 0., -0., -0., -0., 0., 0., -0., 0., 0., 0.,
         -0., -0., -0., 0., -0., -0., 0., 0., 0., 0., -0., 0., 0., 0., -0., -0., -0., -0., 0., 0., -0., -0., -0., -0.,
         -0., 0., 0., 0., -0., -0., 0., 0., 0., 0., 0., 0., -0., -0., 0., 0., -0., -0., -0., -0., 0., -0., -0., -0.,
         -0., -0., 0., 0., -0., -0., -0., 0., -0., -0., -0., -0., -0., -0., 0., -0., -0., -0., -0., -0., -0., 0., 0., 0.,
         0., -0., -0., -0., -0., -0., -0., 0., -0., 0., 0., 0., -0., -0., -0., 0., 0., 0., -0., -0., 0., 0., 0., 0.,
         -0., -0., -0., -0., 0., 0., -0., -0.]])


# file for memory

In [15]:
test_iter= iter(test_DL)
images, labels = next(test_iter)
images = images.to(DEVICE)
labels = labels.to(DEVICE)
input=input_sub(images)

In [95]:
test_iter= iter(test_DS)
images, labels = next(test_iter)
print(images.shape)
print(labels)
images = images.unsqueeze(0)
images = images.to(DEVICE)
labels=torch.tensor([labels]).to(DEVICE)
labels=labels.unsqueeze(0)
print(images.shape)
print(labels.shape)
input=input_sub(images)

torch.Size([3, 32, 32])
3
torch.Size([1, 3, 32, 32])
torch.Size([1, 1])


In [101]:
#layer1
input1=torch.floor(input / layer_params[0]['S_in']) + layer_params[0]['Z_in']

q_w1=layer_params[0]['q_w']
q_bias1=layer_params[0]['q_bias']
scale_factor1=layer_params[0]['Scale_factor']
Z_out1=layer_params[0]['Z_out']

output_tmp1= quanted_sub_layers[0](input1)
output_tmp1=nn.ReLU()(output_tmp1)
output1= torch.floor(output_tmp1 * scale_factor1*(2**(-16))) + Z_out1

In [ ]:
for i in range(len(layer_params)):
    print(f"Layer {i+1} scale_factor: {layer_params[i]['Scale_factor']}")
    print(f"Layer {i+1} zero_point  : {layer_params[i]['Z_out']}")

Layer 1 scale_factor: 133.0
Layer 1 zero_point  : -128.0
Layer 2 scale_factor: 49.0
Layer 2 zero_point  : -128.0
Layer 3 scale_factor: 21.0
Layer 3 zero_point  : 16.0


In [51]:
input1_0=input1[:,0:128]
input1_1=input1[:,128:256]
input1_2=input1[:,256:384]
input1_3=input1[:,384:512]

print(input1_0[0][:10])
print(input1_1[0][:10])
print(input1_2[0][:10])
print(input1_3[0][:10])

tensor([-127., -127., -116., -127., -122., -111., -128., -128., -119., -127.],
       grad_fn=<SliceBackward0>)
tensor([-128., -127.,  -96.,  -62.,  -73., -123., -127., -121., -119., -119.],
       grad_fn=<SliceBackward0>)
tensor([ -19., -119., -121., -123., -127., -128., -127., -128., -124., -112.],
       grad_fn=<SliceBackward0>)
tensor([-128., -125., -112., -127.,  -86., -108., -128., -122.,  -81., -125.],
       grad_fn=<SliceBackward0>)


In [ ]:
input1=torch.round(input / layer_params[0]['S_in']) + layer_params[0]['Z_in']
input1_0=input1[:,0:128]
print(input1_0)

tensor([[-127., -127., -116., -127., -122., -111., -128., -128., -119., -127.,
         -128., -122., -124., -128., -127., -118.,  -30., -128., -122., -127.,
         -128., -109., -128., -125.,  -86., -128., -128., -125., -119.,  -95.,
          -67., -126., -103., -128., -113., -117., -128., -128., -116., -118.,
         -110., -123., -127., -123., -128., -121., -128., -108., -124., -126.,
         -128.,  -95., -125., -128., -122., -123., -126., -102., -128., -128.,
         -128., -128., -115., -111., -120., -113.,  -98.,  -91., -100., -122.,
         -128., -128., -117., -128., -126., -127., -128., -111., -108., -128.,
          -94., -128., -127., -127., -127., -102.,  -94., -128., -127., -126.,
         -128., -107., -121., -128., -128., -128., -124., -128., -100.,  -48.,
         -123., -128., -119., -128., -127., -102., -128., -128.,  -93., -128.,
         -126., -128., -123., -128., -128.,  -98., -128., -126., -128., -128.,
          -82., -128., -128., -123., -128., -128.,  

In [102]:
q_w1=layer_params[0]['q_w']
q_bias1=layer_params[0]['q_bias']
scale_factor1=layer_params[0]['Scale_factor']
Z_out1=layer_params[0]['Z_out']

output_tmp1= quanted_sub_layers[0](input1)
output_tmp1=nn.ReLU()(output_tmp1)
output1= torch.floor(output_tmp1 * scale_factor1*(2**(-16))) + Z_out1

output1_0=output1[:,0:128]
print(output1_0)

tensor([[ -66., -128.,   34., -128., -128.,  -30.,  -11., -128., -128., -128.,
         -127., -128., -128., -128.,  -27., -128., -128., -128., -128., -128.,
         -128., -126., -128., -128., -128., -128.,  -76., -108., -128.,  -15.,
         -116.,  -14., -126., -128., -128., -128.,   -6.,   23.,  -46., -106.,
         -128., -128.,  -64.,  -50., -128., -128.,  -82., -128.,  -54., -119.,
         -128., -128., -128.,  -15.,  -87., -117., -128., -128.,  -66., -128.,
          -49., -128., -119., -128.,   40., -103., -128., -128.,  -18.,  -55.,
            2., -128.,  -11., -128., -112.,  -28., -128., -128.,  -84., -128.,
         -128., -128., -128., -128., -128., -113., -128.,   -6., -128., -128.,
         -119.,  -55., -128.,  -51.,  -94.,   -2., -128., -128., -128., -115.,
         -127., -111.,    1.,  -28., -128., -117., -128.,   -5., -128., -128.,
         -128., -128., -128.,  -83., -128., -128., -113., -121., -128.,  -28.,
         -128., -125., -128., -128.,   27., -128., -

In [103]:
output_tmp1= input1@layer_params[0]['q_w'].T
print(output_tmp1[0][:128])

tensor([-144651.,       0., -145084.,       0., -150027., -205556., -244064.,
              0.,       0.,       0., -224044., -302873., -276146.,       0.,
        -199807.,       0.,       0.,       0., -141070.,       0., -120859.,
        -226837.,       0.,       0., -254349., -230191., -195886., -218865.,
         -11113., -154258., -140540., -175570., -180774.,       0.,       0.,
              0., -202686., -191001., -109655., -215471., -175945.,       0.,
        -193301., -176166.,       0., -245393.,   -7357.,       0., -173579.,
        -173237., -202138.,       0.,       0., -217605., -159576., -114281.,
              0.,       0., -218566.,       0., -171747.,       0.,  -71681.,
        -176639., -167569., -130889., -156805.,       0., -142770., -136330.,
        -194142., -226360., -128379.,       0.,  -91172., -200476.,       0.,
              0., -105788.,       0.,       0.,       0.,       0.,       0.,
              0., -131701.,       0., -107745.,       0.,       

In [104]:
output_tmp1= input1@layer_params[0]['q_w'].T
output_tmp1=output_tmp1 + layer_params[0]['q_bias']
output_fin1=nn.ReLU()(output_tmp1)
output_fin2= torch.floor(output_fin1 * layer_params[0]['Scale_factor']*(2**(-16))) + layer_params[0]['Z_out']
print(output_fin2[0][:128])

tensor([ -66., -128.,   34., -128., -128.,  -30.,  -11., -128., -128., -128.,
        -127., -128., -128., -128.,  -27., -128., -128., -128., -128., -128.,
        -128., -126., -128., -128., -128., -128.,  -76., -108., -128.,  -15.,
        -116.,  -14., -126., -128., -128., -128.,   -6.,   23.,  -46., -106.,
        -128., -128.,  -64.,  -50., -128., -128.,  -82., -128.,  -54., -119.,
        -128., -128., -128.,  -15.,  -87., -117., -128., -128.,  -66., -128.,
         -49., -128., -119., -128.,   40., -103., -128., -128.,  -18.,  -55.,
           2., -128.,  -11., -128., -112.,  -28., -128., -128.,  -84., -128.,
        -128., -128., -128., -128., -128., -113., -128.,   -6., -128., -128.,
        -119.,  -55., -128.,  -51.,  -94.,   -2., -128., -128., -128., -115.,
        -127., -111.,    1.,  -28., -128., -117., -128.,   -5., -128., -128.,
        -128., -128., -128.,  -83., -128., -128., -113., -121., -128.,  -28.,
        -128., -125., -128., -128.,   27., -128., -128.,  -59.],

In [66]:
output_tmp1_0=output_tmp1[:,0:128]
output_tmp1_1=output_tmp1[:,128:256]
output_tmp1_2=output_tmp1[:,256:384]
output_tmp1_3=output_tmp1[:,384:512]

print(output_tmp1_0[0][:10])
print(torch.floor(output_tmp1_0*scale_factor1*(2**(-16))+Z_out1)[0][:10])
print(output_tmp1_1[0][:10])
print(output_tmp1_2[0][:10])
print(output_tmp1_3[0][:10])

tensor([31400.,     0., 81084.,     0.,   247., 49154., 58543.,     0.,     0.,
            0.], grad_fn=<SliceBackward0>)
tensor([ -65., -128.,   36., -128., -128.,  -29.,  -10., -128., -128., -128.],
       grad_fn=<SliceBackward0>)
tensor([  483.,     0.,     0., 54673.,  8925., 59908.,     0., 55130.,     0.,
        72796.], grad_fn=<SliceBackward0>)
tensor([    0., 78417.,     0.,     0.,     0.,     0., 12859.,  1705.,  6262.,
            0.], grad_fn=<SliceBackward0>)
tensor([    0., 68736.,     0.,     0.,     0.,     0., 57197.,     0., 34431.,
            0.], grad_fn=<SliceBackward0>)


In [33]:
print(output1.shape)
output1_0=output1[:,0:128]
output1_1=output1[:,128:256]
output1_2=output1[:,256:384]
output1_3=output1[:,384:512]

print(output1_0[0][:10])
print(output1_1[0][:10])
print(output1_2[0][:10])
print(output1_3[0][:10])

torch.Size([1, 512])
tensor([ -65., -128.,   36., -128., -128.,  -29.,  -10., -128., -128., -128.],
       grad_fn=<SliceBackward0>)
tensor([-128., -128., -128.,  -18., -110.,   -7., -128.,  -17., -128.,   19.],
       grad_fn=<SliceBackward0>)
tensor([-128.,   31., -128., -128., -128., -128., -102., -125., -116., -128.],
       grad_fn=<SliceBackward0>)
tensor([-128.,   11., -128., -128., -128., -128.,  -12., -128.,  -59., -128.],
       grad_fn=<SliceBackward0>)


In [81]:
r_split = 128
c_split = 128

q_w1_splits = []
for i in range(4):
    sub_matrix=[]
    for j in range(4):
        row_start = i * r_split
        row_end = (i + 1) * r_split
        col_start = j * c_split
        col_end = (j + 1) * c_split
        
        sub_matrix.append(q_w1[row_start:row_end, col_start:col_end])
    q_w1_splits.append(sub_matrix)

q_bias1_splits = []
for i in range(4):
    row_start = i * r_split
    row_end = (i + 1) * r_split
    q_bias1_splits.append(q_bias1[row_start:row_end])

input1_splits= []
for i in range(4):
    col_start = i * c_split
    col_end = (i + 1) * c_split
    input1_splits.append(input1[:, col_start:col_end])

output1_splits = []
for i in range(4):
    row_start = i * r_split
    row_end = (i + 1) * r_split
    output1_splits.append(output1[:, row_start:row_end])


In [82]:
output_tmp1_00 = input1_splits[0].to(DEVICE) @ q_w1_splits[0][0].t().to(DEVICE)
output_tmp1_01 = input1_splits[1].to(DEVICE) @ q_w1_splits[0][1].t().to(DEVICE) + output_tmp1_00[0].to(DEVICE)
output_tmp1_02 = input1_splits[2].to(DEVICE) @ q_w1_splits[0][2].t().to(DEVICE) + output_tmp1_01[0].to(DEVICE)
output_tmp1_03 = input1_splits[3].to(DEVICE) @ q_w1_splits[0][3].t().to(DEVICE) + output_tmp1_02[0].to(DEVICE)
output_tmp1_fin1= output_tmp1_03 + q_bias1_splits[0].to(DEVICE)
output_tmp1_fin2=nn.ReLU()(output_tmp1_fin1)
print(output_tmp1_3.shape)

torch.Size([1, 128])


In [94]:
output_tmp1_00 = input1_splits[0].to(DEVICE) @ q_w1_splits[0][0].t().to(DEVICE)
output_tmp1_01 = input1_splits[1].to(DEVICE) @ q_w1_splits[0][1].t().to(DEVICE) + output_tmp1_00[0].to(DEVICE)
output_tmp1_02 = input1_splits[2].to(DEVICE) @ q_w1_splits[0][2].t().to(DEVICE) + output_tmp1_01[0].to(DEVICE)
output_tmp1_03 = input1_splits[3].to(DEVICE) @ q_w1_splits[0][3].t().to(DEVICE) + output_tmp1_02[0].to(DEVICE)
output_tmp1_fin1= output_tmp1_03 + q_bias1_splits[0].to(DEVICE)
output_tmp1_fin2=nn.ReLU()(output_tmp1_fin1)
output_tmp1_fin3= torch.floor(output_tmp1_fin2 * scale_factor1*(2**(-16))) + Z_out1
print(output_tmp1_fin3[0][:])

tensor([ -65., -128.,   36., -128., -128.,  -29.,  -10., -128., -128., -128.,
        -127., -128., -128., -128.,  -25., -128., -128., -128., -128., -128.,
        -127., -125., -128., -128., -128., -128.,  -74., -106., -128.,  -13.,
        -115.,  -12., -126., -128., -128., -128.,   -3.,   25.,  -45., -104.,
        -128., -128.,  -62.,  -48., -128., -128.,  -82., -128.,  -53., -118.,
        -127., -128., -128.,  -13.,  -87., -116., -128., -128.,  -64., -128.,
         -48., -128., -118., -128.,   42., -102., -128., -128.,  -17.,  -54.,
           4., -127.,   -9., -128., -112.,  -27., -128., -128.,  -83., -128.,
        -128., -128., -128., -128., -128., -112., -128.,   -5., -128., -128.,
        -119.,  -55., -128.,  -50.,  -92.,    0., -128., -128., -128., -114.,
        -126., -110.,    3.,  -27., -128., -117., -128.,   -3., -128., -128.,
        -128., -128., -128.,  -81., -128., -128., -112., -121., -128.,  -26.,
        -128., -125., -128., -128.,   30., -128., -128.,  -56.],

In [84]:
print(output_tmp1_00[0][:10])
print(output_tmp1_01[0][:10])
print(output_tmp1_02[0][:10])
print(output_tmp1_03[0][-5:])

tensor([-11698.,      0., -52212.,      0., -43285., -57384., -59922.,      0.,
             0.,      0.], grad_fn=<SliceBackward0>)
tensor([ -50129.,       0.,  -83943.,       0.,  -77713., -105107., -123169.,
              0.,       0.,       0.], grad_fn=<SliceBackward0>)
tensor([-108608.,       0., -106122.,       0., -114877., -158567., -185123.,
              0.,       0.,       0.], grad_fn=<SliceBackward0>)
tensor([      0., -273635.,       0.,       0., -217468.],
       grad_fn=<SliceBackward0>)


In [70]:
print(input1_splits[0][0][:10])
print(output_tmp1_03[0][:10])
print(output1_splits[0][0][:10])

tensor([-127., -127., -116., -127., -122., -111., -128., -128., -119., -127.],
       grad_fn=<SliceBackward0>)
tensor([-35235.,  78417., -38038.,      0., -34967., -46466., -45339.,   1705.,
          6262.,      0.], grad_fn=<SliceBackward0>)
tensor([ -65., -128.,   36., -128., -128.,  -29.,  -10., -128., -128., -128.],
       grad_fn=<SliceBackward0>)


In [60]:
print(output_tmp1_fin2[0][:10])
print((output_tmp1_fin2*layer_params[0]['Scale_factor']*(2**(-16)))[0][:10])
print((torch.floor(output_tmp1_fin2*layer_params[0]['Scale_factor']*(2**(-16)))+layer_params[0]['Z_out'])[0][10:20])

tensor([31400.,     0., 81084.,     0.,   247., 49154., 58543.,     0.,     0.,
            0.], grad_fn=<SliceBackward0>)
tensor([ 63.7238,   0.0000, 164.5534,   0.0000,   0.5013,  99.7541, 118.8083,
          0.0000,   0.0000,   0.0000], grad_fn=<SliceBackward0>)
tensor([-127., -128., -128., -128.,  -25., -128., -128., -128., -128., -128.],
       grad_fn=<SliceBackward0>)


In [21]:
print(output_tmp1_0.shape)
print(input1_splits[0][0][:10].data)
print(q_w1_splits[0][0][0][:10])
print((input1_splits[0][0]@q_w1_splits[0][0].t())[:10])
print(q_bias1_splits[0][:10])
print(output_tmp1_0[0][:10])

torch.Size([1, 128])
tensor([-127., -127., -116., -127., -122., -111., -128., -128., -118., -127.])
tensor([  0.,  -0.,   0.,  -0., -47.,   0.,   0.,   0.,   0.,   0.])
tensor([-11698.,      0., -52212.,      0., -43285., -57384., -59922.,      0.,
             0.,      0.], grad_fn=<SliceBackward0>)
tensor([175247.,      0., 225250.,      0., 150093., 254199., 301878.,      0.,
             0.,      0.])
tensor([-11698.,      0., -52212.,      0., -43285., -57384., -59922.,      0.,
             0.,      0.], grad_fn=<SliceBackward0>)


In [22]:
print(q_w1_splits[0][0].shape)
print(q_bias1_splits[0].shape)
print(input1_splits[0].shape)
print(output_tmp1_0.shape)

torch.Size([128, 128])
torch.Size([128])
torch.Size([1, 128])
torch.Size([1, 128])


In [71]:
print(input1_splits[0][0][:10])
print(q_w1_splits[0][0][1][:10])
print(q_bias1_splits[0][:10])
print(output_tmp1_3[0][:10])
# print(output_tmp1_0[0][:])
print((torch.round(output_tmp1_fin2*scale_factor1*(2**(-16))) + Z_out1))

tensor([-127., -127., -116., -127., -122., -111., -128., -128., -119., -127.],
       grad_fn=<SliceBackward0>)
tensor([-0., 0., -0., -0., 0., -0., 0., 0., 0., -0.])
tensor([175243.,      0., 225244.,      0., 150091., 254187., 301864.,      0.,
             0.,      0.])
tensor([    0., 68736.,     0.,     0.,     0.,     0., 57197.,     0., 34431.,
            0.], grad_fn=<SliceBackward0>)
tensor([[ 228.,   11.,  329., -128.,  177.,  388.,  601., -128.,  -58., -128.,
          328.,  525.,  431., -128.,  379., -128.,   51., -128.,  240.,  -70.,
          139.,  335., -128.,  -26.,  363.,  311.,  322.,  337., -105.,  340.,
          169.,  413.,  242., -121., -128., -128.,  460.,  411.,  327.,  332.,
          228., -128.,  329.,  366., -128.,  366.,   79., -128.,  374.,  233.,
          373., -128., -128.,  432.,  237.,  207., -121., -128.,  397., -128.,
          419., -128.,   35.,  229.,  380.,  266.,  190., -128.,  272.,  302.,
          429.,  332.,  251., -126.,   73.,  379., 

In [72]:
print(scale_factor1)

tensor(133.)


In [112]:
print(output_tmp1_3[0][-10:])

output_tmp1_fin1= output_tmp1_3 + q_bias1_splits[0].to(DEVICE)
print(output_tmp1_fin1[0][-10:])

output_tmp1_fin2=nn.ReLU()(output_tmp1_fin1)
print(output_tmp1_fin2[0][-10:])

output_tmp1_fin3= output_tmp1_fin2 * scale_factor1
print(output_tmp1_fin3[0][-10:])

output_tmp1_fin4= torch.floor(output_tmp1_fin3*(2**(-16))) + Z_out1
print(output_tmp1_fin4[0][-10:])




tensor([      0., -157632., -250755., -156631., -137797.,       0., -273203.,
              0.,       0., -217317.], grad_fn=<SliceBackward0>)
tensor([    0., 50843.,  -913.,  1854., -3511.,     0., 78294.,     0.,     0.,
        35652.], grad_fn=<SliceBackward0>)
tensor([    0., 50843.,     0.,  1854.,     0.,     0., 78294.,     0.,     0.,
        35652.], grad_fn=<SliceBackward0>)
tensor([       0.,  6609590.,        0.,   241020.,        0.,        0.,
        10178220.,        0.,        0.,  4634760.], grad_fn=<SliceBackward0>)
tensor([-128.,  -28., -128., -125., -128., -128.,   27., -128., -128.,  -58.],
       grad_fn=<SliceBackward0>)


In [114]:
print(q_bias1_splits[0][-10:])

tensor([     0., 208475., 249842., 158485., 134286.,      0., 351497.,      0.,
             0., 252969.])


In [154]:

# convert to int lists and print non-zero with indices
def print_int_and_nonzero(tensor, name):
    print(f"{name}")
    arr = tensor.detach().cpu().to(torch.int64).numpy()
    print(f"{arr.tolist()}")
    nz_idx = np.nonzero(arr)[0].tolist()
    pairs = [(int(i), int(arr[i])) for i in nz_idx]
    # print(f"non-zero (idx, val): {pairs}")

# print_int_and_nonzero(output_tmp1_0[0][10:30], "output_tmp1_0")
# print_int_and_nonzero(output_tmp1_0[0][110:], "output_tmp1_0")

# print_int_and_nonzero(output_tmp1_1[0][:10], "output_tmp1_0")
# print_int_and_nonzero(output_tmp1_1[0][100:110], "output_tmp1_0")

# print_int_and_nonzero(output_tmp1_2[0][:10], "output_tmp1_0")
# print_int_and_nonzero(output_tmp1_2[0][100:110], "output_tmp1_0")

# print_int_and_nonzero(output_tmp1_3[0][:10], "output_tmp1_0")
# print_int_and_nonzero(output_tmp1_3[0][100:110], "output_tmp1_0")
# print_int_and_nonzero(output_tmp1_3[0][110:120], "output_tmp1_0")
# print_int_and_nonzero(output_tmp1_3[0][120:], "output_tmp1_0")


print_int_and_nonzero(output_tmp1_fin1[0][:10], "output_tmp1_1")
print_int_and_nonzero(output_tmp1_fin2[0][:10], "output_tmp1_2")
print_int_and_nonzero(output_tmp1_fin3[0][:10], "output_tmp1_3")


print_int_and_nonzero(output_tmp1_fin4[0][:10], "output_tmp1_3")
print_int_and_nonzero(output_tmp1_fin4[0][30:50], "output_tmp1_3")
print_int_and_nonzero(output_tmp1_fin4[0][80:100], "output_tmp1_3")
print_int_and_nonzero(output_tmp1_fin4[0][120:], "output_tmp1_3")
# ...existing code...

output_tmp1_1
[31548, 0, 81426, 0, 207, 49447, 58827, 0, 0, 0]
output_tmp1_2
[31548, 0, 81426, 0, 207, 49447, 58827, 0, 0, 0]
output_tmp1_3
[4101240, 0, 10585380, 0, 26910, 6428110, 7647510, 0, 0, 0]
output_tmp1_3
[-66, -128, 33, -128, -128, -30, -12, -128, -128, -128]
output_tmp1_3
[-116, -14, -126, -128, -128, -128, -5, 22, -47, -105, -128, -128, -64, -50, -128, -128, -82, -128, -54, -119]
output_tmp1_3
[-128, -128, -128, -128, -128, -112, -128, -7, -128, -128, -119, -56, -128, -52, -93, -2, -128, -128, -128, -114]
output_tmp1_3
[-128, -125, -128, -128, 27, -128, -128, -58]


In [85]:
# csr 변환
print(q_w1_splits[0][0].shape)

val_list=[]
col_idx_list=[]
row_num_list=[]
nnz=[]
for j in range(4):
    val=[]
    col_idx=[]
    row_num=[]
    for i in range(q_w1_splits[0][j].shape[0]):
        row = q_w1_splits[0][j][i]
        indices = torch.nonzero(row).flatten().tolist()
        values = row[indices].tolist()
        
        row_num.append(len(values))
        val.extend(values)
        col_idx.extend(indices)
        # print(f"Row {i}: Indices: {indices}, Values: {values}")
        # print(values)
        # print(f"Row {i}: row_num: {len(values)}")
    val_list.append(val)
    col_idx_list.append(col_idx)
    row_num_list.append(row_num)
    nnz.append(len(val))


val=[]
col_idx=[]
row_num=[]
for i in range(4):
    val.extend(val_list[i])
    col_idx.extend(col_idx_list[i])
    row_num.extend(row_num_list[i])
total_nnz=len(val)

    




torch.Size([128, 128])


In [86]:
print(len(val))
print(nnz)
print(sum(nnz))
print(len(col_idx))
print(len(row_num))

3628
[933, 907, 891, 897]
3628
3628
512


In [26]:
print(len(val_list))
print(nnz)
for i in range(len(val_list)):
    print(f"Sub-matrix {i}:")
    print(f"Values: {len(val_list[i])}")
    print(f"Column Indices: {len(col_idx_list[i])}")
    print(f"Row Num: {len(row_num_list[i])}")
    print(sum(row_num_list[i]))

4
[933, 907, 891, 897]
Sub-matrix 0:
Values: 933
Column Indices: 933
Row Num: 128
933
Sub-matrix 1:
Values: 907
Column Indices: 907
Row Num: 128
907
Sub-matrix 2:
Values: 891
Column Indices: 891
Row Num: 128
891
Sub-matrix 3:
Values: 897
Column Indices: 897
Row Num: 128
897


In [27]:
# print(row_num)
# print(sum(row_num))

In [28]:
print(len(val))
print(len(col_idx))
print(len(row_num))

3628
3628
512


In [29]:
print(max(col_idx))

127


In [30]:
print(val[0])
print(col_idx[0])

print(format(int(val[0]) & 0xFF, "08b"))
print(format(int(val[0]) & 0xFFFF, "016b"))

print(format(col_idx[0], "07b"))

print(format(int(val[0]) & 0xFF, "08b")+format(col_idx[0], "07b"))

-47.0
4
11010001
1111111111010001
0000100
110100010000100


In [31]:
print(val[:10])
print(val[-1])

print(col_idx[:10])

print(input1_splits[0][0][:10].data)
print(input1_splits[0][0][4].item())

[-47.0, 23.0, 37.0, 24.0, 39.0, 29.0, 40.0, 57.0, 42.0, 50.0]
46.0
[4, 47, 62, 67, 112, 122, 4, 16, 24, 29]
tensor([-127., -127., -116., -127., -122., -111., -128., -128., -118., -127.])
-122.0


In [47]:
print(row_num[0])
print(val[:6])
print(col_idx[:6])
print([input1_splits[0][0][i].item() for i in col_idx[:6]])
sum_=0
for i in range(6):
    v = val[i]
    c = col_idx[i]
    inp = input1_splits[0][0][c].item()
    mult = v * inp
    print(f"val: {v}, input: {inp}, mult: {mult}")
    sum_ += mult
print(f"Sum: {sum_}, q_bias: {q_bias1_splits[0][0].item()}, Total: {sum_ + q_bias1_splits[0][0].item()}")

6
[-47.0, 23.0, 37.0, 24.0, 39.0, 29.0]
[4, 47, 62, 67, 112, 122]
[-122.0, -108.0, -115.0, -91.0, -123.0, -128.0]
val: -47.0, input: -122.0, mult: 5734.0
val: 23.0, input: -108.0, mult: -2484.0
val: 37.0, input: -115.0, mult: -4255.0
val: 24.0, input: -91.0, mult: -2184.0
val: 39.0, input: -123.0, mult: -4797.0
val: 29.0, input: -128.0, mult: -3712.0
Sum: -11698.0, q_bias: 175247.0, Total: 163549.0


In [64]:
print(val[932])
print(nnz)
print(val[932:932+10])
print(val[0])
print(val[-1])


45.0
[933, 907, 891, 897]
[45.0, 21.0, 40.0, 24.0, 32.0, 32.0, 27.0, 25.0, 34.0, 36.0]
-47.0
46.0


In [72]:
print(output_tmp1_0[0][:10])
print(output_tmp1_1[0][:10])
print(output_tmp1_2[0][:10])
print(output_tmp1_3[0][:10])

tensor([-11698.,      0., -52212.,      0., -43285., -57384., -59922.,      0.,
             0.,      0.], grad_fn=<SliceBackward0>)
tensor([ -50076.,       0.,  -83775.,       0.,  -77713., -104937., -123011.,
              0.,       0.,       0.], grad_fn=<SliceBackward0>)
tensor([-108497.,       0., -105912.,       0., -114919., -158369., -184896.,
              0.,       0.,       0.], grad_fn=<SliceBackward0>)
tensor([-143699.,       0., -143824.,       0., -149886., -204752., -243051.,
              0.,       0.,       0.], grad_fn=<SliceBackward0>)


In [88]:
print(row_num[128])
print(val[933])
print(col_idx[933:946])
print([input1_splits[0][0][i].item() for i in col_idx[933:946]])
sum_=0
for i in range(933,946):
    v = val[i]
    c = col_idx[i]
    inp = input1_splits[1][0][c].item()
    mult = v * inp
    print(f"val: {v}, input: {inp}, mult: {mult}")
    sum_ += mult
print(f"Sum: {sum_}, output_tmp: {output_tmp1_0[0][0].item()}, Total: {sum_ + output_tmp1_0[0][0].item()}")

13
21.0
[4, 20, 24, 30, 67, 69, 75, 83, 89, 93, 113, 118, 126]
[-122.0, -128.0, -86.0, -67.0, -91.0, -122.0, -127.0, -127.0, -126.0, -128.0, -128.0, -128.0, -91.0]
val: 21.0, input: -73.0, mult: -1533.0
val: 40.0, input: -117.0, mult: -4680.0
val: 24.0, input: -102.0, mult: -2448.0
val: 32.0, input: -65.0, mult: -2080.0
val: 32.0, input: -127.0, mult: -4064.0
val: 27.0, input: -111.0, mult: -2997.0
val: 25.0, input: -107.0, mult: -2675.0
val: 34.0, input: -127.0, mult: -4318.0
val: 36.0, input: -114.0, mult: -4104.0
val: 38.0, input: -115.0, mult: -4370.0
val: -21.0, input: -128.0, mult: 2688.0
val: 25.0, input: -86.0, mult: -2150.0
val: 50.0, input: -114.0, mult: -5700.0
Sum: -38431.0, output_tmp: 31400.0, Total: -7031.0


In [90]:
print(row_num[256:260])
print(val[933])
print(col_idx[933:946])
print([input1_splits[0][0][i].item() for i in col_idx[933:946]])
sum_=0
for i in range(933,946):
    v = val[i]
    c = col_idx[i]
    inp = input1_splits[1][0][c].item()
    mult = v * inp
    print(f"val: {v}, input: {inp}, mult: {mult}")
    sum_ += mult
print(f"Sum: {sum_}, output_tmp: {output_tmp1_0[0][0].item()}, Total: {sum_ + output_tmp1_0[0][0].item()}")

[17, 0, 9, 0]
21.0
[4, 20, 24, 30, 67, 69, 75, 83, 89, 93, 113, 118, 126]
[-122.0, -128.0, -86.0, -66.0, -91.0, -122.0, -127.0, -127.0, -126.0, -128.0, -128.0, -128.0, -91.0]
val: 21.0, input: -72.0, mult: -1512.0
val: 40.0, input: -117.0, mult: -4680.0
val: 24.0, input: -102.0, mult: -2448.0
val: 32.0, input: -64.0, mult: -2048.0
val: 32.0, input: -127.0, mult: -4064.0
val: 27.0, input: -111.0, mult: -2997.0
val: 25.0, input: -107.0, mult: -2675.0
val: 34.0, input: -127.0, mult: -4318.0
val: 36.0, input: -114.0, mult: -4104.0
val: 38.0, input: -115.0, mult: -4370.0
val: -21.0, input: -128.0, mult: 2688.0
val: 25.0, input: -86.0, mult: -2150.0
val: 50.0, input: -114.0, mult: -5700.0
Sum: -38378.0, output_tmp: -11698.0, Total: -50076.0


In [137]:
print(row_num[256+107])
print(val[933])
print(col_idx[933:946])
print([input1_splits[0][0][i].item() for i in col_idx[933:946]])
sum_=0
for i in range(933,946):
    v = val[i]
    c = col_idx[i]
    inp = input1_splits[1][0][c].item()
    mult = v * inp
    print(f"val: {v}, input: {inp}, mult: {mult}")
    sum_ += mult
print(f"Sum: {sum_}, output_tmp: {output_tmp1_0[0][0].item()}, Total: {sum_ + output_tmp1_0[0][0].item()}")

13
21.0
[4, 20, 24, 30, 67, 69, 75, 83, 89, 93, 113, 118, 126]
[-122.0, -128.0, -86.0, -66.0, -91.0, -122.0, -127.0, -127.0, -126.0, -128.0, -128.0, -128.0, -91.0]
val: 21.0, input: -72.0, mult: -1512.0
val: 40.0, input: -117.0, mult: -4680.0
val: 24.0, input: -102.0, mult: -2448.0
val: 32.0, input: -64.0, mult: -2048.0
val: 32.0, input: -127.0, mult: -4064.0
val: 27.0, input: -111.0, mult: -2997.0
val: 25.0, input: -107.0, mult: -2675.0
val: 34.0, input: -127.0, mult: -4318.0
val: 36.0, input: -114.0, mult: -4104.0
val: 38.0, input: -115.0, mult: -4370.0
val: -21.0, input: -128.0, mult: 2688.0
val: 25.0, input: -86.0, mult: -2150.0
val: 50.0, input: -114.0, mult: -5700.0
Sum: -38378.0, output_tmp: -11698.0, Total: -50076.0


In [132]:
print(nnz)
print(row_num[256:260])
print(val[933])
print(col_idx[933:946])

[933, 907, 891, 897]
[17, 0, 9, 0]
21.0
[4, 20, 24, 30, 67, 69, 75, 83, 89, 93, 113, 118, 126]


In [79]:
#encoding
# val은 2의보수 8비트
# col_idx는 7비트 양수
# row_num는 7비트 양수
# 15비트 val[i-1]col_idx[i]을 반복
qw_data_mem=[]

for j in range(4):
    qw_data_mem_sub=[]
    N=nnz[j]
    N_prev=sum(nnz[:j])
    print(N)
    for i in range(nnz[j]):
        
        
        if i==0:
            data=format(int(val[N_prev+N-1]) & 0xFF, "08b")+format(col_idx[N_prev+i], "07b")
            # data=f"{int(val[N-1])},{col_idx[i]}"
            print(int(val[N-1]))
        else:
            data=format(int(val[N_prev+i-1]) & 0xFF, "08b")+format(col_idx[N_prev+i], "07b")
            # data=f"{int(val[i-1])},{col_idx[i]}"
        qw_data_mem_sub.append(data)
    qw_data_mem.extend(qw_data_mem_sub)

row_num_mem=[]
for i in range(len(row_num)):
    data=format(row_num[i], "07b")
    row_num_mem.append(data)

q_bias_mem=[]
for i in range(q_bias1_splits[0].shape[0]):
    data=format(int(q_bias1_splits[0][i].item()) & 0xFFFFFFFF, "032b")
    q_bias_mem.append(data)

input_mem=[]
for j in range(4):
    input_mem_split=[]
    for i in range(input1_splits[0].shape[1]):

        data=format(int(input1_splits[j][0][i].item()) & 0xFF, "08b")
        input_mem_split.append(data)
    input_mem.append(input_mem_split)

output_tmp_mem=[]
for i in range(output_tmp1_0.shape[1]):
    data=format(int(output_tmp1_0[0][i].item()) & 0xFFFFFFFF, "032b")
    output_tmp_mem.append(data)

933
45
907
38
891
92
897
33


In [70]:
print(len(qw_data_mem))
print(val[0])
print(qw_data_mem[1])

3628
-47.0
110100010101111


In [34]:
print(len(input_mem[1]))

128


In [35]:
print(output_tmp1_0[0][-10:])

tensor([     0., -21810., -68809., -33716., -38927.,      0., -70895.,      0.,
             0., -61342.], grad_fn=<SliceBackward0>)


In [36]:
print(output_tmp1_3.shape)
print(output_tmp1_3[0].shape)

print(input1_splits[0].shape)
print(input1_splits[0][0][0].item())

print(q_bias1_splits[0][0])

torch.Size([1, 128])
torch.Size([128])
torch.Size([1, 128])
-127.0
tensor(175247.)


In [77]:
print(len(qw_data_mem))
print(qw_data_mem[933:946])

print(len(row_num_mem))
print(row_num_mem[:5])

print(len(q_bias_mem))
print(q_bias_mem[:5])

print(len(input_mem))
print(input_mem[:5])

print(len(output_tmp_mem))
print(output_tmp_mem[:5])

3628
['110101100000100', '110100010101111', '000101110111110', '001001011000011', '000110001110000', '001001111111010', '000111010000100', '001010000010000', '001110010011000', '001010100011101', '001100100110011', '001000000110111', '001110100111001']
512
['0000110', '0000000', '0010000', '0000000', '0001011']
128
['00000000000000101010110010001111', '00000000000000000000000000000000', '00000000000000110110111111100010', '00000000000000000000000000000000', '00000000000000100100101001001101']
4
[['10000001', '10000001', '10001100', '10000001', '10000110', '10010001', '10000000', '10000000', '10001010', '10000001', '10000000', '10000110', '10000100', '10000000', '10000001', '10001010', '11100010', '10000000', '10000110', '10000001', '10000000', '10010011', '10000000', '10000011', '10101010', '10000000', '10000000', '10000011', '10001001', '10100001', '10111110', '10000010', '10011001', '10000000', '10001111', '10001011', '10000000', '10000000', '10001100', '10001010', '10010010', '10000

In [83]:
sim_dir=os.path.join(path, 'sim_file')

# qw_data_mem (len=933, 15-bit strings)
with open(os.path.join(sim_dir, 'qw_data_mem.coe'), 'w', encoding='utf-8') as f:
    f.write("memory_initialization_radix=2;\nmemory_initialization_vector=\n")
    f.write(",\n".join(qw_data_mem) + ";")

# row_num_mem (len=128, 7-bit strings)
with open(os.path.join(sim_dir, 'row_num_mem.coe'), 'w', encoding='utf-8') as f:
    f.write("memory_initialization_radix=2;\nmemory_initialization_vector=\n")
    f.write(",\n".join(row_num_mem) + ";")

# q_bias_mem (len=128, 32-bit strings)
with open(os.path.join(sim_dir, 'q_bias_mem.coe'), 'w', encoding='utf-8') as f:
    f.write("memory_initialization_radix=2;\nmemory_initialization_vector=\n")
    f.write(",\n".join(q_bias_mem) + ";")

# input_mem (len=128, 8-bit strings)
for j in range(4):
    with open(os.path.join(sim_dir, f'input_mem_{j}.coe'), 'w', encoding='utf-8') as f:
        f.write("memory_initialization_radix=2;\nmemory_initialization_vector=\n")
        f.write(",\n".join(input_mem[j]) + ";")

# output_tmp_mem (len=128, 32-bit strings)
with open(os.path.join(sim_dir, 'output_tmp_mem.coe'), 'w', encoding='utf-8') as f:
    f.write("memory_initialization_radix=2;\nmemory_initialization_vector=\n")
    f.write(",\n".join(output_tmp_mem) + ";")

In [39]:
qw_data_mem_combined_h = [format(int(bstr, 2), "04x") for bstr in qw_data_mem]
row_num_mem_h = [format(int(bstr, 2), "02x") for bstr in row_num_mem]
q_bias_mem_h = [format(int(bstr, 2), "08x") for bstr in q_bias_mem]
input_mem_h = [format(int(bstr, 2), "02x") for bstr in input_mem]
output_tmp_mem_h = [format(int(bstr, 2), "08x") for bstr in output_tmp_mem]


TypeError: int() can't convert non-string with explicit base

In [ ]:
print(input1_splits[0][0][47])
print(input_mem[47])
print(input_mem_h[47])
print(q_bias1_splits[0][0])
print(q_bias_mem[0])
print(q_bias_mem_h[0])

tensor(-108., grad_fn=<SelectBackward0>)
10010100
94
tensor(175240.)
00000000000000101010110010001000
0002ac88


In [ ]:
print(format(int(input_mem[47], 2), "02x"))

94


In [ ]:
print("qw_data_mem (hex):")
print(len(qw_data_mem_combined_h))
print(qw_data_mem_combined_h[:5])
print("row_num_mem (hex):")
print(len(row_num_mem_h))
print(row_num_mem_h[:5])
print("q_bias_mem (hex):")
print(len(q_bias_mem_h))
print(q_bias_mem_h[:10])
print("input_mem (hex):")
print(len(input_mem_h))
print(input_mem_h[47])
print("output_tmp_mem (hex):")
print(len(output_tmp_mem_h))
print(output_tmp_mem_h[:5])

qw_data_mem (hex):
933
['1684', '68af', '0bbe', '12c3', '0c70']
row_num_mem (hex):
128
['06', '00', '10', '00', '0b']
q_bias_mem (hex):
128
['0002ac88', '00000000', '00036fd8', '00000000', '00024a4a', '0003e0e3', '00049b1e', '00000000', '00000000', '00000000']
input_mem (hex):
128
94
output_tmp_mem (hex):
128
['ffffd24e', '00000000', 'ffff33c5', '00000000', 'ffff56eb']


In [167]:
# CHANNEL_GROUP=0
for CHANNEL_GROUP in range(4):
    val_list=[]
    col_idx_list=[]
    row_num_list=[]
    nnz=[]
    for j in range(4):
        val=[]
        col_idx=[]
        row_num=[]
        for i in range(q_w1_splits[CHANNEL_GROUP][j].shape[0]):
            row = q_w1_splits[CHANNEL_GROUP][j][i]
            indices = torch.nonzero(row).flatten().tolist()
            values = row[indices].tolist()
            
            row_num.append(len(values))
            val.extend(values)
            col_idx.extend(indices)
            # print(f"Row {i}: Indices: {indices}, Values: {values}")
            # print(values)
            # print(f"Row {i}: row_num: {len(values)}")
        val_list.append(val)
        col_idx_list.append(col_idx)
        row_num_list.append(row_num)
        nnz.append(len(val))
    print(nnz)

[933, 907, 891, 897]
[827, 831, 786, 830]
[831, 847, 831, 873]
[724, 713, 670, 716]


In [164]:
# csr 변환
# channel 128~255

CHANNEL_GROUP=5

val_list=[]
col_idx_list=[]
row_num_list=[]
nnz=[]
for j in range(4):
    val=[]
    col_idx=[]
    row_num=[]
    for i in range(q_w1_splits[CHANNEL_GROUP][j].shape[0]):
        row = q_w1_splits[CHANNEL_GROUP][j][i]
        indices = torch.nonzero(row).flatten().tolist()
        values = row[indices].tolist()
        
        row_num.append(len(values))
        val.extend(values)
        col_idx.extend(indices)
        # print(f"Row {i}: Indices: {indices}, Values: {values}")
        # print(values)
        # print(f"Row {i}: row_num: {len(values)}")
    val_list.append(val)
    col_idx_list.append(col_idx)
    row_num_list.append(row_num)
    nnz.append(len(val))

val=[]
col_idx=[]
row_num=[]
for i in range(4):
    val.extend(val_list[i])
    col_idx.extend(col_idx_list[i])
    row_num.extend(row_num_list[i])
total_nnz=len(val)

#encoding
# val은 2의보수 8비트
# col_idx는 7비트 양수
# row_num는 7비트 양수
# 15비트 val[i-1]col_idx[i]을 반복
qw_data_mem=[]

for j in range(4):
    qw_data_mem_sub=[]
    N=nnz[j]
    N_prev=sum(nnz[:j])
    print(N)
    for i in range(nnz[j]):
        
        
        if i==0:
            data=format(int(val[N_prev+N-1]) & 0xFF, "08b")+format(col_idx[N_prev+i], "07b")
            # data=f"{int(val[N-1])},{col_idx[i]}"
            print(int(val[N-1]))
        else:
            data=format(int(val[N_prev+i-1]) & 0xFF, "08b")+format(col_idx[N_prev+i], "07b")
            # data=f"{int(val[i-1])},{col_idx[i]}"
        qw_data_mem_sub.append(data)
    qw_data_mem.extend(qw_data_mem_sub)

row_num_mem=[]
for i in range(len(row_num)):
    data=format(row_num[i], "07b")
    row_num_mem.append(data)

q_bias_mem=[]
for i in range(q_bias1_splits[CHANNEL_GROUP].shape[0]):
    data=format(int(q_bias1_splits[CHANNEL_GROUP][i].item()) & 0xFFFFFFFF, "032b")
    q_bias_mem.append(data)



sim_dir=os.path.join(path, 'sim_file')

# qw_data_mem (len=933, 15-bit strings)
with open(os.path.join(sim_dir, f'qw_data_mem_{CHANNEL_GROUP}.coe'), 'w', encoding='utf-8') as f:
    f.write("memory_initialization_radix=2;\nmemory_initialization_vector=\n")
    f.write(",\n".join(qw_data_mem) + ";")

# row_num_mem (len=128, 7-bit strings)
with open(os.path.join(sim_dir, f'row_num_mem_{CHANNEL_GROUP}.coe'), 'w', encoding='utf-8') as f:
    f.write("memory_initialization_radix=2;\nmemory_initialization_vector=\n")
    f.write(",\n".join(row_num_mem) + ";")

# q_bias_mem (len=128, 32-bit strings)
with open(os.path.join(sim_dir, f'q_bias_mem_{CHANNEL_GROUP}.coe'), 'w', encoding='utf-8') as f:
    f.write("memory_initialization_radix=2;\nmemory_initialization_vector=\n")
    f.write(",\n".join(q_bias_mem) + ";")


IndexError: list index out of range

In [160]:
print(q_w1_splits[0][0].shape)

torch.Size([128, 128])


# file for memory 2


## 준비

In [30]:
test_iter= iter(test_DS)
images, labels = next(test_iter)
print(images.shape)
print(labels)
images = images.unsqueeze(0)
images = images.to(DEVICE)
labels=torch.tensor([labels]).to(DEVICE)
labels=labels.unsqueeze(0)
print(images.shape)
print(labels.shape)
input=input_sub(images)

torch.Size([3, 32, 32])
3
torch.Size([1, 3, 32, 32])
torch.Size([1, 1])


In [31]:
#layer1
input1=torch.floor(input / layer_params[0]['S_in']) + layer_params[0]['Z_in']

q_w1=layer_params[0]['q_w']
q_bias1=layer_params[0]['q_bias']
scale_factor1=layer_params[0]['Scale_factor']
Z_out1=layer_params[0]['Z_out']

output_tmp1= quanted_sub_layers[0](input1)
output_tmp1=nn.ReLU()(output_tmp1)
output1= torch.floor(output_tmp1 * scale_factor1*(2**(-16))) + Z_out1

# input1_0=input1[:,0:128]
# input1_1=input1[:,128:256]
# input1_2=input1[:,256:384]
# input1_3=input1[:,384:512]

In [32]:
r_split = 128
c_split = 128

q_w1_splits = []
for i in range(4):
    sub_matrix=[]
    for j in range(4):
        row_start = i * r_split
        row_end = (i + 1) * r_split
        col_start = j * c_split
        col_end = (j + 1) * c_split
        
        sub_matrix.append(q_w1[row_start:row_end, col_start:col_end])
    q_w1_splits.append(sub_matrix)

q_bias1_splits = []
for i in range(4):
    row_start = i * r_split
    row_end = (i + 1) * r_split
    q_bias1_splits.append(q_bias1[row_start:row_end])

input1_splits= []
for i in range(4):
    col_start = i * c_split
    col_end = (i + 1) * c_split
    input1_splits.append(input1[:, col_start:col_end])

output1_splits = []
for i in range(4):
    row_start = i * r_split
    row_end = (i + 1) * r_split
    output1_splits.append(output1[:, row_start:row_end])


In [33]:
def csr_gen_sub(weight_submatrix):
    val=[]
    col_idx=[]
    row_num=[]
    # print(weight_submatrix.shape)
    for i in range(weight_submatrix.shape[0]):
        row = weight_submatrix[i]
        indices = torch.nonzero(row).flatten().tolist()
        values = row[indices].tolist()
        
        row_num.append(len(values))
        val.extend(values)
        col_idx.extend(indices)
    nnz=len(val)
    return val, col_idx, row_num, nnz

In [34]:
def csr_gen(weight_matrix_list):
    val_list=[]
    col_idx_list=[]
    row_num_list=[]
    nnzs=[]
    for j in range(len(weight_matrix_list)):
        val, col_idx, row_num, nnz = csr_gen_sub(weight_matrix_list[j])
        val_list.append(val)
        col_idx_list.append(col_idx)
        row_num_list.append(row_num)
        nnzs.append(nnz)
    return val_list, col_idx_list, row_num_list, nnzs
    # val=[]
    # col_idx=[]
    # row_num=[]
    # for i in range(len(weight_matrix_list)):
    #     val.extend(val_list[i])
    #     col_idx.extend(col_idx_list[i])
    #     row_num.extend(row_num_list[i])
    # total_nnz=len(val)
    # return val, col_idx, row_num, total_nnz, nnzs



In [35]:
#encoding 함수
def encode_vector(val, bit_width):
    encoded=[]
    for v in val:
        data=format(int(v) & (2**bit_width-1), f"0{bit_width}b")
        encoded.append(data)
    return encoded

def encode_csr_sub(val, col_idx, row_num):
    qw_data_mem=[]
    for i in range(len(val)):
        if i==0:
            data=format(int(val[-1]) & 0xFF, "08b")+format(col_idx[i], "07b")
        else:
            data=format(int(val[i-1]) & 0xFF, "08b")+format(col_idx[i], "07b")
        qw_data_mem.append(data)
    row_num_mem=[]
    for i in range(len(row_num)):
        data=format(row_num[i], "07b")
        row_num_mem.append(data)
    return qw_data_mem, row_num_mem

def encode_csr(val_list, col_idx_list, row_num_list):
    qw_data_mem=[]
    row_num_mem=[]
    for i in range(len(val_list)):
        qw_data_sub, row_num_sub = encode_csr_sub(val_list[i], col_idx_list[i], row_num_list[i])
        qw_data_mem.extend(qw_data_sub)
        row_num_mem.extend(row_num_sub)
    return qw_data_mem, row_num_mem

In [36]:
def bin_list_to_hex(bin_list, width_hex=0, upper=True):
    fmt = "X" if upper else "x"
    out = []
    for b in bin_list:
        h = format(int(b, 2), fmt)
        if width_hex:
            h = h.zfill(width_hex)
        out.append(h)
    return out




In [37]:
#input encoding
input_encoded=[]
for j in range(4):
    input_encoded.append(encode_vector(input1_splits[j][0].tolist(), 8))

In [38]:
input_encoded_h=[]
for j in range(4):
    input_encoded_h.append(bin_list_to_hex(input_encoded[j], width_hex=2))

In [39]:
i=0

print(input1_splits[i][0][:])
print(input_encoded[i][:10])
print(input_encoded_h[i][:10])

tensor([-127., -127., -116., -128., -123., -111., -128., -128., -119., -127.,
        -128., -122., -124., -128., -127., -118.,  -30., -128., -122., -128.,
        -128., -110., -128., -125.,  -86., -128., -128., -126., -120.,  -95.,
         -66., -126., -103., -128., -114., -117., -128., -128., -116., -118.,
        -110., -124., -128., -123., -128., -122., -128., -108., -124., -127.,
        -128.,  -95., -126., -128., -123., -123., -127., -102., -128., -128.,
        -128., -128., -116., -111., -121., -114.,  -98.,  -91., -101., -122.,
        -128., -128., -118., -128., -126., -128., -128., -111., -109., -128.,
         -94., -128., -128., -128., -128., -103.,  -94., -128., -128., -127.,
        -128., -107., -121., -128., -128., -128., -124., -128., -100.,  -48.,
        -123., -128., -120., -128., -128., -102., -128., -128.,  -93., -128.,
        -127., -128., -124., -128., -128.,  -99., -128., -127., -128., -128.,
         -82., -128., -128., -124., -128., -128.,  -92., -127.],

In [89]:
print(sim_dir)
print(os.listdir(sim_dir))

g:\내 드라이브\02_Education\02_Research_and_Projects\2512_2602_Dash Research Intern\pytorch_practice\sim_file
['input_mem_0.mem', 'input_mem_1.mem', 'input_mem_2.mem', 'input_mem_3.mem', 'qw_data_mem_0.mem', 'row_num_mem_0.mem', 'q_bias_mem_0.mem', 'qw_data_mem_1.mem', 'row_num_mem_1.mem', 'q_bias_mem_1.mem', 'qw_data_mem_2.mem', 'row_num_mem_2.mem', 'q_bias_mem_2.mem', 'qw_data_mem_3.mem', 'row_num_mem_3.mem', 'q_bias_mem_3.mem']


In [91]:
sim_dir=os.path.join(path, 'sim_file')
# sim_dir=r'C:\Users\darwin5991\Desktop\vivado_project\NPU\NPU.srcs\sources_1\memory'


In [92]:
# input_mem (len=128, 8-bit strings)
for i in range(4):
    # with open(os.path.join(sim_dir, f'input_mem_{i}.coe'), 'w', encoding='utf-8') as f:
    #     f.write("memory_initialization_radix=2;\nmemory_initialization_vector=\n")
    #     f.write(",\n".join(input_encoded[i]) + ";")

    
    with open(os.path.join(sim_dir, f'input_mem_{i}.mem'), 'w', encoding='utf-8') as f:
        f.write("\n".join(input_encoded_h[i]) + "\n")


## for 128x128

In [116]:
#for tb_ (128x128)
output_tmp1_00 = input1_splits[0].to(DEVICE) @ q_w1_splits[0][0].t().to(DEVICE)

In [143]:
val, col_idx, row_num, nnz = csr_gen_sub(q_w1_splits[0][0])
print(f"Values: {len(val)}")
print(f"Column Indices: {len(col_idx)}")
print(f"Row Numbers: {len(row_num)}")
print(row_num[:10])
print(f"Non-zero Elements: {nnz}")

torch.Size([128, 128])
Values: 933
Column Indices: 933
Row Numbers: 128
[6, 0, 16, 0, 11, 18, 20, 0, 0, 0]
Non-zero Elements: 933


In [118]:
print(input1_splits[0].shape)
print(input1_splits[0][0][:10].data)
print(q_w1_splits[0][0][0][:10])
print((input1_splits[0][0]@q_w1_splits[0][0].t())[:10])
print(output_tmp1_00[0][:10])

torch.Size([1, 128])
tensor([-127., -128., -116., -128., -123., -112., -128., -128., -119., -127.])
tensor([  0.,  -0.,   0.,  -0., -47.,   0.,   0.,   0.,   0.,   0.])
tensor([-11798.,      0., -53077.,      0., -43484., -57572., -60144.,      0.,
             0.,      0.], grad_fn=<SliceBackward0>)
tensor([-11798.,      0., -53077.,      0., -43484., -57572., -60144.,      0.,
             0.,      0.], grad_fn=<SliceBackward0>)


In [146]:
print(val[:6])
print(col_idx[:6])
for i in col_idx[:6]:
    print(input1_splits[0][0][i].item())

[-47.0, 23.0, 37.0, 24.0, 39.0, 29.0]
[4, 47, 62, 67, 112, 122]
-123.0
-109.0
-116.0
-93.0
-124.0
-128.0


In [119]:
# weight, output encoding
qw_data_mem, row_num_mem = encode_csr_sub(val, col_idx, row_num)
output_tmp1_00_encoded = encode_vector(output_tmp1_00[0].tolist(), 32)

qw_data_mem_h = bin_list_to_hex(qw_data_mem, width_hex=4)
row_num_mem_h = bin_list_to_hex(row_num_mem, width_hex=2)
output_tmp1_00_h  = bin_list_to_hex(output_tmp1_00_encoded,  width_hex=8)


print(len(qw_data_mem))
print(len(row_num_mem))
print(len(output_tmp1_00_encoded))
print("------------------dec------------------")
print(val[:10])
print(col_idx[:10])
print(row_num[:10])
print(output_tmp1_00[0][:10])
print("------------------bin------------------")
print(qw_data_mem[:10])
print(row_num_mem[:10])
print(output_tmp1_00_encoded[:10])
print("------------------hex------------------")
print(qw_data_mem_h[:10])
print(row_num_mem_h[:10])
print(output_tmp1_00_h[:10])


933
128
128
------------------dec------------------
[-47.0, 23.0, 37.0, 24.0, 39.0, 29.0, 40.0, 57.0, 42.0, 50.0]
[4, 47, 62, 67, 112, 122, 4, 16, 24, 29]
[6, 0, 16, 0, 11, 18, 20, 0, 0, 0]
tensor([-11798.,      0., -53077.,      0., -43484., -57572., -60144.,      0.,
             0.,      0.], grad_fn=<SliceBackward0>)
------------------bin------------------
['001011010000100', '110100010101111', '000101110111110', '001001011000011', '000110001110000', '001001111111010', '000111010000100', '001010000010000', '001110010011000', '001010100011101']
['0000110', '0000000', '0010000', '0000000', '0001011', '0010010', '0010100', '0000000', '0000000', '0000000']
['11111111111111111101000111101010', '00000000000000000000000000000000', '11111111111111110011000010101011', '00000000000000000000000000000000', '11111111111111110101011000100100', '11111111111111110001111100011100', '11111111111111110001010100010000', '00000000000000000000000000000000', '00000000000000000000000000000000', '000000000

In [120]:
# for i in range(10):
#     print(output_tmp1_00_h[i*10:i*10+10])

for i in range(3):
    print(output_tmp1_00_h[100+i*10:100+(i+1)*10])

['FFFF1BC9', 'FFFF1903', 'FFFF340E', 'FFFEDC8D', '00000000', 'FFFF79CC', '00000000', 'FFFF21E2', '00000000', '00000000']
['FFFF6862', '00000000', 'FFFF73A8', 'FFFF6E8B', '00000000', '00000000', 'FFFF9E91', 'FFFF2AC7', '00000000', 'FFFFA9C9']
['FFFEF2A1', 'FFFF7C14', 'FFFF6767', '00000000', 'FFFEE886', '00000000', '00000000', 'FFFF0EB8']


In [122]:
qw_data_mem_h = bin_list_to_hex(qw_data_mem, width_hex=4)  # 15bit → 4 hex
row_num_mem_h = bin_list_to_hex(row_num_mem, width_hex=2)
# q_bias_mem_h  = bin_list_to_hex(q_bias_mem,  width_hex=8)

In [ ]:
# qw_data_mem (len=933, 15-bit strings)
with open(os.path.join(sim_dir, 'qw_data_mem.coe'), 'w', encoding='utf-8') as f:
    f.write("memory_initialization_radix=2;\nmemory_initialization_vector=\n")
    f.write(",\n".join(qw_data_mem) + ";")

# row_num_mem (len=128, 7-bit strings)
with open(os.path.join(sim_dir, 'row_num_mem.coe'), 'w', encoding='utf-8') as f:
    f.write("memory_initialization_radix=2;\nmemory_initialization_vector=\n")
    f.write(",\n".join(row_num_mem) + ";")

# # output_tmp_mem (len=128, 32-bit strings)
# with open(os.path.join(sim_dir, 'output_tmp_mem.coe'), 'w', encoding='utf-8') as f:
#     f.write("memory_initialization_radix=2;\nmemory_initialization_vector=\n")
#     f.write(",\n".join(output_tmp_mem) + ";")

## for 128x512

In [ ]:
#for tb_ (128x512)
output_tmp1_00 = input1_splits[0].to(DEVICE) @ q_w1_splits[0][0].t().to(DEVICE)
output_tmp1_01 = input1_splits[1].to(DEVICE) @ q_w1_splits[0][1].t().to(DEVICE) + output_tmp1_00[0].to(DEVICE)
output_tmp1_02 = input1_splits[2].to(DEVICE) @ q_w1_splits[0][2].t().to(DEVICE) + output_tmp1_01[0].to(DEVICE)
output_tmp1_03 = input1_splits[3].to(DEVICE) @ q_w1_splits[0][3].t().to(DEVICE) + output_tmp1_02[0].to(DEVICE)
output_tmp1_fin1= output_tmp1_03 + q_bias1_splits[0].to(DEVICE)
output_tmp1_fin2=nn.ReLU()(output_tmp1_fin1)
output_tmp1_fin3= torch.floor(output_tmp1_fin2 * scale_factor1*(2**(-16))) + Z_out1
output_tmp1_fin4 = torch.clamp(output_tmp1_fin3, q_min_, q_max_).to(torch.int8)

In [ ]:
for j in range(len(q_w1_splits[0])):
    val, col_idx, row_num, nnz = csr_gen_sub(q_w1_splits[0][j])
    print(nnz)


torch.Size([128, 128])
933
torch.Size([128, 128])
907
torch.Size([128, 128])
891
torch.Size([128, 128])
897


In [ ]:
val_list, col_idx_list, row_num_list, nnzs= csr_gen(q_w1_splits[0])

print(f"Total Non-zero Elements: {sum(nnzs)}")
print(f"Non-zero Elements per Sub-matrix: {nnzs}")
print(f"Values: {[len(v) for v in val_list]}")
print(f"Column Indices: {[len(c) for c in col_idx_list]}")
print(f"Row Numbers: {[len(r) for r in row_num_list]}")

torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
Total Non-zero Elements: 3628
Non-zero Elements per Sub-matrix: [933, 907, 891, 897]
Values: [933, 907, 891, 897]
Column Indices: [933, 907, 891, 897]
Row Numbers: [128, 128, 128, 128]


In [ ]:
#encoding
qw_data_mem, row_num_mem= encode_csr(val_list, col_idx_list, row_num_list)
qw_data_mem_h = bin_list_to_hex(qw_data_mem, width_hex=4)
row_num_mem_h = bin_list_to_hex(row_num_mem, width_hex=2)

output_tmp1_00_encoded = encode_vector(output_tmp1_00[0].tolist(), 32)
output_tmp1_00_h  = bin_list_to_hex(output_tmp1_00_encoded,  width_hex=8)
output_tmp1_01_encoded = encode_vector(output_tmp1_01[0].tolist(), 32)
output_tmp1_01_h  = bin_list_to_hex(output_tmp1_01_encoded,  width_hex=8)
output_tmp1_02_encoded = encode_vector(output_tmp1_02[0].tolist(), 32)
output_tmp1_02_h  = bin_list_to_hex(output_tmp1_02_encoded,  width_hex=8)
output_tmp1_03_encoded = encode_vector(output_tmp1_03[0].tolist(), 32)
output_tmp1_03_h  = bin_list_to_hex(output_tmp1_03_encoded,  width_hex=8)



3628 3628
512 512


In [ ]:
print(len(qw_data_mem),len(qw_data_mem_h))
print(len(row_num_mem),len(row_num_mem_h))
print(len(output_tmp1_00_encoded), len(output_tmp1_00_h))
print(len(output_tmp1_01_encoded), len(output_tmp1_01_h))
print(len(output_tmp1_02_encoded), len(output_tmp1_02_h))
print(len(output_tmp1_03_encoded), len(output_tmp1_03_h))
print("------------------dec------------------")
for i in range(4):
    print(f"Sub-matrix {i}:")
    print(val_list[i][:10])
    print(col_idx_list[i][:10])
    print(row_num_list[i][:10])
    print(nnzs[i])
print("------------------bin------------------")
for i in range(4):
    print(f"Sub-matrix {i}:")
    if i==0:
        print(qw_data_mem[:10])
        print(row_num_mem[:10])
        print(output_tmp1_00_encoded[0:10])
    else:
        start_idx = sum(nnzs[:i])
        print(qw_data_mem[start_idx:start_idx+10])
        print(row_num_mem[i*128:i*128+10])
        if i==1: print(output_tmp1_01_encoded[:10])
        elif i==2: print(output_tmp1_02_encoded[:10])
        else : print(output_tmp1_03_encoded[:10])
print("------------------hex------------------")
for i in range(4):
    print(f"Sub-matrix {i}:")
    if i==0:
        print(qw_data_mem_h[:10])
        print(row_num_mem_h[:10])
        print(output_tmp1_00_h[0:10])
    else:
        start_idx = sum(nnzs[:i])
        print(qw_data_mem_h[start_idx:start_idx+10])
        print(row_num_mem_h[i*128:i*128+10])
        if i==1: print(output_tmp1_01_h[:10])
        elif i==2: print(output_tmp1_02_h[:10])
        else : print(output_tmp1_03_h[:10])

3628 3628
512 512
128 128
128 128
128 128
128 128
------------------dec------------------
Sub-matrix 0:
[-47.0, 23.0, 37.0, 24.0, 39.0, 29.0, 40.0, 57.0, 42.0, 50.0]
[4, 47, 62, 67, 112, 122, 4, 16, 24, 29]
[6, 0, 16, 0, 11, 18, 20, 0, 0, 0]
933
Sub-matrix 1:
[21.0, 40.0, 24.0, 32.0, 32.0, 27.0, 25.0, 34.0, 36.0, 38.0]
[4, 20, 24, 30, 67, 69, 75, 83, 89, 93]
[13, 0, 17, 0, 9, 14, 18, 0, 0, 0]
907
Sub-matrix 2:
[46.0, -22.0, 49.0, 27.0, 22.0, 36.0, 37.0, 23.0, 43.0, 27.0]
[3, 13, 21, 22, 35, 36, 37, 41, 54, 57]
[17, 0, 9, 0, 9, 13, 15, 0, 0, 0]
891
Sub-matrix 3:
[44.0, 27.0, 31.0, 40.0, 33.0, -30.0, 36.0, -19.0, 41.0, 33.0]
[4, 12, 16, 24, 41, 43, 74, 76, 80, 102]
[14, 0, 14, 0, 10, 14, 17, 0, 0, 0]
897
------------------bin------------------
Sub-matrix 0:
['001011010000100', '110100010101111', '000101110111110', '001001011000011', '000110001110000', '001001111111010', '000111010000100', '001010000010000', '001110010011000', '001010100011101']
['0000110', '0000000', '0010000', '0000000'

In [ ]:
i=0
print(nnzs)
start_idx = sum(nnzs[:i])
print("------------------dec------------------")
print(input1_splits[0][0][:10])
print(val_list[i][:10])
print(col_idx_list[i][:10])
print(row_num_list[i][:10])
print(output_tmp1_00[0][:10])

print("------------------bin------------------")
print(input_encoded[0][:10])
print(qw_data_mem[start_idx:start_idx+10])
print(row_num_mem[i*128:i*128+10])
print(output_tmp1_00_encoded[:10])

print("------------------hex------------------")
print(input_encoded_h[0][:10])
print(qw_data_mem_h[start_idx:start_idx+10])
print(row_num_mem_h[i*128:i*128+10])
print(output_tmp1_00_h[:10])

[933, 907, 891, 897]
------------------dec------------------
tensor([-127., -128., -116., -128., -123., -112., -128., -128., -119., -127.],
       grad_fn=<SliceBackward0>)
[-47.0, 23.0, 37.0, 24.0, 39.0, 29.0, 40.0, 57.0, 42.0, 50.0]
[4, 47, 62, 67, 112, 122, 4, 16, 24, 29]
[6, 0, 16, 0, 11, 18, 20, 0, 0, 0]
tensor([-11798.,      0., -53077.,      0., -43484., -57572., -60144.,      0.,
             0.,      0.], grad_fn=<SliceBackward0>)
------------------bin------------------
['10000001', '10000000', '10001100', '10000000', '10000101', '10010000', '10000000', '10000000', '10001001', '10000001']
['001011010000100', '110100010101111', '000101110111110', '001001011000011', '000110001110000', '001001111111010', '000111010000100', '001010000010000', '001110010011000', '001010100011101']
['0000110', '0000000', '0010000', '0000000', '0001011', '0010010', '0010100', '0000000', '0000000', '0000000']
['11111111111111111101000111101010', '00000000000000000000000000000000', '1111111111111111001

In [158]:
j=9
print(output_tmp1_03[0][j*10:j*10+10])
print(output_tmp1_03[0][j*10+10:j*10+20])
print(output_tmp1_03[0][j*10+20:j*10+30])
print(output_tmp1_03[0][j*10+30:j*10+40])

tensor([-183052., -122705.,       0., -143689., -184115., -202653.,       0.,
        -165367.,       0., -192586.], grad_fn=<SliceBackward0>)
tensor([-179293., -118765., -173657., -207024.,       0.,  -53414.,       0.,
        -209866.,       0.,       0.], grad_fn=<SliceBackward0>)
tensor([-139931.,       0., -178161., -186406.,       0.,       0., -132131.,
        -148049.,       0., -160202.], grad_fn=<SliceBackward0>)
tensor([-250973., -156967., -137817.,       0., -276811.,       0.,       0.,
        -219308.], grad_fn=<SliceBackward0>)


In [ ]:
print(output_tmp1_fin3[0][:10])
print(output_tmp1_fin4[0][:10])

tensor([ -67., -128.,   32., -128., -128.,  -31.,  -13., -128., -128., -128.],
       grad_fn=<SliceBackward0>)
tensor([ -67, -128,   32, -128, -128,  -31,  -13, -128, -128, -128],
       dtype=torch.int8)


In [ ]:
print(len(q_bias1_splits))
print(q_bias1_splits[0].shape)
print(type(q_bias1_splits[0].tolist()))
print(len(q_bias1_splits[0].tolist()))

4
torch.Size([128])
<class 'list'>
128


In [ ]:
q_bias_mem_encoded=encode_vector(q_bias1_splits[0].tolist(), 32)
q_bias_mem_h  = bin_list_to_hex(q_bias_mem_encoded,  width_hex=8)

In [ ]:
print(scale_factor1)
print(Z_out1)

tensor(135.)
tensor(-128.)


In [ ]:
# qw_data_mem (len=933, 15-bit strings)
with open(os.path.join(sim_dir, 'qw_data_mem.coe'), 'w', encoding='utf-8') as f:
    f.write("memory_initialization_radix=2;\nmemory_initialization_vector=\n")
    f.write(",\n".join(qw_data_mem) + ";")

# row_num_mem (len=128, 7-bit strings)
with open(os.path.join(sim_dir, 'row_num_mem.coe'), 'w', encoding='utf-8') as f:
    f.write("memory_initialization_radix=2;\nmemory_initialization_vector=\n")
    f.write(",\n".join(row_num_mem) + ";")

# q_bias_mem (len=128, 32-bit strings)
with open(os.path.join(sim_dir, 'q_bias_mem.coe'), 'w', encoding='utf-8') as f:
    f.write("memory_initialization_radix=2;\nmemory_initialization_vector=\n")
    f.write(",\n".join(q_bias_mem_encoded) + ";")

    

## for 512x512

In [105]:
for i in range(4):
    # for j in range(len(q_w1_splits[0])):
    #     val, col_idx, row_num, nnz = csr_gen_sub(q_w1_splits[i][j])
    #     # print(nnz)
    val_list, col_idx_list, row_num_list, nnzs= csr_gen(q_w1_splits[i])
    print(nnzs)
    qw_data_mem, row_num_mem= encode_csr(val_list, col_idx_list, row_num_list)
    q_bias_mem_encoded= encode_vector(q_bias1_splits[i].tolist(), 32)

    qw_data_mem_h = bin_list_to_hex(qw_data_mem, width_hex=4)  # 15bit → 4 hex
    row_num_mem_h = bin_list_to_hex(row_num_mem, width_hex=2)
    q_bias_mem_h  = bin_list_to_hex(q_bias_mem_encoded,  width_hex=8)

    # # qw_data_mem (len=933, 15-bit strings)
    # with open(os.path.join(sim_dir, f'qw_data_mem_{i}.coe'), 'w', encoding='utf-8') as f:
    #     f.write("memory_initialization_radix=2;\nmemory_initialization_vector=\n")
    #     f.write(",\n".join(qw_data_mem) + ";")

    # # row_num_mem (len=128, 7-bit strings)
    # with open(os.path.join(sim_dir, f'row_num_mem_{i}.coe'), 'w', encoding='utf-8') as f:
    #     f.write("memory_initialization_radix=2;\nmemory_initialization_vector=\n")
    #     f.write(",\n".join(row_num_mem) + ";")

    # # q_bias_mem (len=128, 32-bit strings)
    # with open(os.path.join(sim_dir, f'q_bias_mem_{i}.coe'), 'w', encoding='utf-8') as f:
    #     f.write("memory_initialization_radix=2;\nmemory_initialization_vector=\n")
    #     f.write(",\n".join(q_bias_mem_encoded) + ";")
    
    # qw_data_mem (len=933, 15-bit strings)
    with open(os.path.join(sim_dir, f'qw_data_mem_{i}.mem'), 'w', encoding='utf-8') as f:
        # f.write("\n".join(qw_data_mem) + "\n")
        f.write("\n".join(qw_data_mem_h) + "\n")


    # row_num_mem (len=128, 7-bit strings)
    with open(os.path.join(sim_dir, f'row_num_mem_{i}.mem'), 'w', encoding='utf-8') as f:
        # f.write("\n".join(row_num_mem) + "\n")
        f.write("\n".join(row_num_mem_h) + "\n")

    # q_bias_mem (len=128, 32-bit strings)
    with open(os.path.join(sim_dir, f'q_bias_mem_{i}.mem'), 'w', encoding='utf-8') as f:
        # f.write("\n".join(q_bias_mem_encoded) + "\n")
        f.write("\n".join(q_bias_mem_h) + "\n")




[933, 907, 891, 897]
[827, 831, 786, 830]
[831, 847, 831, 873]
[724, 713, 670, 716]


In [59]:
print(output1_splits[0][0][:10])
print(output1_splits[1][0][:10])
print(output1_splits[2][0][:10])
print(output1_splits[3][0][:10])

# output_tmp1= quanted_sub_layers[0](input1)
output_tmp1=input1.to(DEVICE) @ layer_params[0]['q_w'].t().to(DEVICE)
output_tmp11= output_tmp1 + layer_params[0]['q_bias'].to(DEVICE)
output_tmp111=nn.ReLU()(output_tmp11)

output1= torch.floor(output_tmp111 * scale_factor1*(2**(-16))) + Z_out1

tensor([ -70., -128.,   25., -128., -128.,  -35.,  -18., -128., -128., -128.],
       grad_fn=<SliceBackward0>)
tensor([-128., -128., -128.,  -25., -113.,  -15., -128.,  -24., -128.,    9.],
       grad_fn=<SliceBackward0>)
tensor([-128.,   20., -128., -128., -128., -128., -105., -126., -118., -128.],
       grad_fn=<SliceBackward0>)
tensor([-128.,    1., -128., -128., -128., -128.,  -20., -128.,  -63., -128.],
       grad_fn=<SliceBackward0>)


In [60]:
PE=0
i=0
print(input1_splits[PE][0][i*10:(i+1)*10])
print(input1_splits[PE][0][(i+1)*10:(i+2)*10])
print(input1_splits[PE][0][(i+2)*10:(i+3)*10])
print(input1_splits[PE][0][(i+3)*10:(i+4)*10])

tensor([-127., -127., -116., -128., -123., -111., -128., -128., -119., -127.],
       grad_fn=<SliceBackward0>)
tensor([-128., -122., -124., -128., -127., -118.,  -30., -128., -122., -128.],
       grad_fn=<SliceBackward0>)
tensor([-128., -110., -128., -125.,  -86., -128., -128., -126., -120.,  -95.],
       grad_fn=<SliceBackward0>)
tensor([ -66., -126., -103., -128., -114., -117., -128., -128., -116., -118.],
       grad_fn=<SliceBackward0>)


In [61]:
#for tb_ (128x512) PE0

output_tmp1_00 = input1_splits[0].to(DEVICE) @ q_w1_splits[0][0].t().to(DEVICE)
output_tmp1_01 = input1_splits[1].to(DEVICE) @ q_w1_splits[0][1].t().to(DEVICE) + output_tmp1_00[0].to(DEVICE)
output_tmp1_02 = input1_splits[2].to(DEVICE) @ q_w1_splits[0][2].t().to(DEVICE) + output_tmp1_01[0].to(DEVICE)
output_tmp1_03 = input1_splits[3].to(DEVICE) @ q_w1_splits[0][3].t().to(DEVICE) + output_tmp1_02[0].to(DEVICE)
output_tmp1_fin1= output_tmp1_03 + q_bias1_splits[0].to(DEVICE)
output_tmp1_fin2=nn.ReLU()(output_tmp1_fin1)
output_tmp1_fin3= torch.floor(output_tmp1_fin2 * scale_factor1*(2**(-16))) + Z_out1
output_tmp1_fin4 = torch.clamp(output_tmp1_fin3, q_min_, q_max_).to(torch.int8)

output_tmp1={}
output_tmp1['PE0']=[]
output_tmp1['PE0'].append(output_tmp1_00)
output_tmp1['PE0'].append(output_tmp1_01)
output_tmp1['PE0'].append(output_tmp1_02)
output_tmp1['PE0'].append(output_tmp1_03)
output_tmp1['PE0'].append(output_tmp1_fin1)
output_tmp1['PE0'].append(output_tmp1_fin2)
output_tmp1['PE0'].append(output_tmp1_fin3)
output_tmp1['PE0'].append(output_tmp1_fin4)

In [62]:
#for tb_ (128x512) PE1
output_tmp1_11 = input1_splits[1].to(DEVICE) @ q_w1_splits[1][1].t().to(DEVICE)
output_tmp1_12 = input1_splits[2].to(DEVICE) @ q_w1_splits[1][2].t().to(DEVICE) + output_tmp1_11[0].to(DEVICE)
output_tmp1_13 = input1_splits[3].to(DEVICE) @ q_w1_splits[1][3].t().to(DEVICE) + output_tmp1_12[0].to(DEVICE)
output_tmp1_10 = input1_splits[0].to(DEVICE) @ q_w1_splits[1][0].t().to(DEVICE) + output_tmp1_13[0].to(DEVICE)
output_tmp1_fin1= output_tmp1_10 + q_bias1_splits[1].to(DEVICE)
output_tmp1_fin2=nn.ReLU()(output_tmp1_fin1)
output_tmp1_fin3= torch.floor(output_tmp1_fin2 * scale_factor1*(2**(-16))) + Z_out1
output_tmp1_fin4 = torch.clamp(output_tmp1_fin3, q_min_, q_max_).to(torch.int8)
output_tmp1['PE1']=[]
output_tmp1['PE1'].append(output_tmp1_11)
output_tmp1['PE1'].append(output_tmp1_12)
output_tmp1['PE1'].append(output_tmp1_13)
output_tmp1['PE1'].append(output_tmp1_10)
output_tmp1['PE1'].append(output_tmp1_fin1)
output_tmp1['PE1'].append(output_tmp1_fin2)
output_tmp1['PE1'].append(output_tmp1_fin3)
output_tmp1['PE1'].append(output_tmp1_fin4)

In [63]:
#for tb_ (128x512) PE2
output_tmp1_22 = input1_splits[2].to(DEVICE) @ q_w1_splits[2][2].t().to(DEVICE)
output_tmp1_23 = input1_splits[3].to(DEVICE) @ q_w1_splits[2][3].t().to(DEVICE) + output_tmp1_22[0].to(DEVICE)
output_tmp1_20 = input1_splits[0].to(DEVICE) @ q_w1_splits[2][0].t().to(DEVICE) + output_tmp1_23[0].to(DEVICE)
output_tmp1_21 = input1_splits[1].to(DEVICE) @ q_w1_splits[2][1].t().to(DEVICE) + output_tmp1_20[0].to(DEVICE)
output_tmp1_fin1= output_tmp1_21 + q_bias1_splits[2].to(DEVICE)
output_tmp1_fin2=nn.ReLU()(output_tmp1_fin1)
output_tmp1_fin3= torch.floor(output_tmp1_fin2 * scale_factor1*(2**(-16))) + Z_out1
output_tmp1_fin4 = torch.clamp(output_tmp1_fin3, q_min_, q_max_).to(torch.int8)
output_tmp1['PE2']=[]
output_tmp1['PE2'].append(output_tmp1_22)
output_tmp1['PE2'].append(output_tmp1_23)
output_tmp1['PE2'].append(output_tmp1_20)
output_tmp1['PE2'].append(output_tmp1_21)
output_tmp1['PE2'].append(output_tmp1_fin1)
output_tmp1['PE2'].append(output_tmp1_fin2)
output_tmp1['PE2'].append(output_tmp1_fin3)
output_tmp1['PE2'].append(output_tmp1_fin4)

In [64]:
#for tb_ (128x512) PE3
output_tmp1_33 = input1_splits[3].to(DEVICE) @ q_w1_splits[3][3].t().to(DEVICE)
output_tmp1_30 = input1_splits[0].to(DEVICE) @ q_w1_splits[3][0].t().to(DEVICE) + output_tmp1_33[0].to(DEVICE)
output_tmp1_31 = input1_splits[1].to(DEVICE) @ q_w1_splits[3][1].t().to(DEVICE) + output_tmp1_30[0].to(DEVICE)
output_tmp1_32 = input1_splits[2].to(DEVICE) @ q_w1_splits[3][2].t().to(DEVICE) + output_tmp1_31[0].to(DEVICE)
output_tmp1_fin1= output_tmp1_32 + q_bias1_splits[3].to(DEVICE)
output_tmp1_fin2=nn.ReLU()(output_tmp1_fin1)
output_tmp1_fin3= torch.floor(output_tmp1_fin2 * scale_factor1*(2**(-16))) + Z_out1
output_tmp1_fin4 = torch.clamp(output_tmp1_fin3, q_min_, q_max_).to(torch.int8)
output_tmp1['PE3']=[]
output_tmp1['PE3'].append(output_tmp1_33)
output_tmp1['PE3'].append(output_tmp1_30)
output_tmp1['PE3'].append(output_tmp1_31)
output_tmp1['PE3'].append(output_tmp1_32)
output_tmp1['PE3'].append(output_tmp1_fin1)
output_tmp1['PE3'].append(output_tmp1_fin2)
output_tmp1['PE3'].append(output_tmp1_fin3)
output_tmp1['PE3'].append(output_tmp1_fin4)

In [123]:
PE='PE0'
q_cnt=3
i=0
print(output_tmp1[PE][q_cnt][0][i*10:(i+1)*10])

tensor([-144076.,       0., -144059.,       0., -150046., -204931., -243266.,
              0.,       0.,       0.], grad_fn=<SliceBackward0>)


In [122]:
#디버깅 
pe=0
q_cnt=3
idx= (pe+q_cnt)%4
print(idx)

val_list, col_idx_list, row_num_list, nnzs= csr_gen(q_w1_splits[pe])
print(f"PE{pe} - Sub-matrix {idx}:")
print(f"  Values: {len(val_list[idx])}")
print(f"  Column Indices: {len(col_idx_list[idx])}")
print(f"  Row Numbers: {len(row_num_list[idx])}")
print(f"  Non-zero Elements: {nnzs}")

3
PE0 - Sub-matrix 3:
  Values: 897
  Column Indices: 897
  Row Numbers: 128
  Non-zero Elements: [933, 907, 891, 897]


In [82]:
print(len(q_w1_splits[0][0]))

print(q_w1_splits[0][0][0])

128
tensor([  0.,  -0.,   0.,  -0., -47.,   0.,   0.,   0.,   0.,   0.,   0.,  -0.,
         -0.,  -0.,   0.,   0.,  -0.,   0.,  -0.,   0.,   0.,   0.,   0.,   0.,
         -0.,  -0.,   0.,   0.,   0.,  -0.,  -0.,  -0.,   0.,   0.,  -0.,   0.,
          0.,  -0.,   0.,   0.,   0.,  -0.,  -0.,  -0.,  -0.,   0.,   0.,  23.,
         -0.,   0.,   0.,   0.,   0.,   0.,   0.,  -0.,   0.,  -0.,   0.,   0.,
          0.,   0.,  37.,   0.,   0.,   0.,  -0.,  24.,  -0.,   0.,   0.,  -0.,
          0.,   0.,  -0.,   0.,   0.,  -0.,   0.,   0.,   0.,  -0.,   0.,   0.,
          0.,   0.,   0.,  -0.,   0.,  -0.,   0.,   0.,   0.,   0.,   0.,   0.,
         -0.,   0.,   0.,   0.,   0.,   0.,  -0.,   0.,  -0.,   0.,  -0.,   0.,
         -0.,  -0.,   0.,  -0.,  39.,   0.,   0.,   0.,  -0.,   0.,  -0.,   0.,
          0.,   0.,  29.,   0.,   0.,  -0.,   0.,   0.])


In [124]:
print(row_num_list[idx][:3])
print(val_list[idx][:row_num_list[idx][0]])
print(col_idx_list[idx][:row_num_list[idx][0]])
print(input1_splits[idx][0][col_idx_list[idx][:row_num_list[idx][0]]])

accum=0
for i in range(row_num_list[idx][0]):
    value=val_list[idx][i]
    input=input1_splits[idx][0][col_idx_list[idx][i]]
    print(f"Value={value}, Input={input}, Product={value * input}")
    accum += value * input
    print(f"Accumulated Sum: {accum}")


[14, 0, 14]
[44.0, 27.0, 31.0, 40.0, 33.0, -30.0, 36.0, -19.0, 41.0, 33.0, 41.0, 38.0, 33.0, 37.0]
[4, 12, 16, 24, 41, 43, 74, 76, 80, 102, 118, 119, 124, 126]
tensor([ -86., -111., -122., -115., -128., -128., -113., -128., -112., -128.,
         -74.,  -97.,  -55.,  -20.], grad_fn=<IndexBackward0>)
Value=44.0, Input=-86.0, Product=-3784.0
Accumulated Sum: -3784.0
Value=27.0, Input=-111.0, Product=-2997.0
Accumulated Sum: -6781.0
Value=31.0, Input=-122.0, Product=-3782.0
Accumulated Sum: -10563.0
Value=40.0, Input=-115.0, Product=-4600.0
Accumulated Sum: -15163.0
Value=33.0, Input=-128.0, Product=-4224.0
Accumulated Sum: -19387.0
Value=-30.0, Input=-128.0, Product=3840.0
Accumulated Sum: -15547.0
Value=36.0, Input=-113.0, Product=-4068.0
Accumulated Sum: -19615.0
Value=-19.0, Input=-128.0, Product=2432.0
Accumulated Sum: -17183.0
Value=41.0, Input=-112.0, Product=-4592.0
Accumulated Sum: -21775.0
Value=33.0, Input=-128.0, Product=-4224.0
Accumulated Sum: -25999.0
Value=41.0, Input=-74.

In [130]:
print(layer_params[0]['Scale_factor'])

tensor(124.)


In [131]:
#디버깅 
PE='PE0'
i=0
scale_factor1=layer_params[0]['Scale_factor']
i_data_tmp=output_tmp1[PE][3][0][i*10:(i+1)*10]
i_data_bias=q_bias1_splits[0][i*10:(i+1)*10]
w_mac_result=i_data_tmp + i_data_bias
w_scaled_result= nn.ReLU()(w_mac_result) * scale_factor1
w_final_result_shifted= torch.floor(w_scaled_result*(2**(-16)))
w_final_result= w_final_result_shifted + Z_out1


for j in range(10):
    print(f"i_data_tmp: {i_data_tmp[j]}, \t\t i_data_bias: {i_data_bias[j]}")
    print(f"MAC Result: {w_mac_result[j]}, \nScaled Result: {w_scaled_result[j]}")
    print(f"Final Result (before shift): {w_final_result_shifted[j]}, \t Final Result: {w_final_result[j]}\n")




i_data_tmp: -144076.0, 		 i_data_bias: 175253.0
MAC Result: 31177.0, 
Scaled Result: 3865948.0
Final Result (before shift): 58.0, 	 Final Result: -70.0

i_data_tmp: 0.0, 		 i_data_bias: 0.0
MAC Result: 0.0, 
Scaled Result: 0.0
Final Result (before shift): 0.0, 	 Final Result: -128.0

i_data_tmp: -144059.0, 		 i_data_bias: 225259.0
MAC Result: 81200.0, 
Scaled Result: 10068800.0
Final Result (before shift): 153.0, 	 Final Result: 25.0

i_data_tmp: 0.0, 		 i_data_bias: 0.0
MAC Result: 0.0, 
Scaled Result: 0.0
Final Result (before shift): 0.0, 	 Final Result: -128.0

i_data_tmp: -150046.0, 		 i_data_bias: 150095.0
MAC Result: 49.0, 
Scaled Result: 6076.0
Final Result (before shift): 0.0, 	 Final Result: -128.0

i_data_tmp: -204931.0, 		 i_data_bias: 254217.0
MAC Result: 49286.0, 
Scaled Result: 6111464.0
Final Result (before shift): 93.0, 	 Final Result: -35.0

i_data_tmp: -243266.0, 		 i_data_bias: 301901.0
MAC Result: 58635.0, 
Scaled Result: 7270740.0
Final Result (before shift): 110.0

## for layer2

In [138]:
#layer2
input2=output1

q_w2=layer_params[1]['q_w']
q_bias2=layer_params[1]['q_bias']
scale_factor2=layer_params[1]['Scale_factor']
Z_out2=layer_params[1]['Z_out']
print(f"Z_out:{Z_out2}, Scale_factor: {scale_factor2}")

output_tmp2= quanted_sub_layers[1](input2)
output_tmp2=nn.ReLU()(output_tmp2)
output2= torch.floor(output_tmp2 * scale_factor2*(2**(-16))) + Z_out2




r_split = 128
c_split = 128

q_w2_splits = []
for i in range(4):
    sub_matrix=[]
    for j in range(4):
        row_start = i * r_split
        row_end = (i + 1) * r_split
        col_start = j * c_split
        col_end = (j + 1) * c_split
        
        sub_matrix.append(q_w2[row_start:row_end, col_start:col_end])
    q_w2_splits.append(sub_matrix)

q_bias2_splits = []
for i in range(4):
    row_start = i * r_split
    row_end = (i + 1) * r_split
    q_bias2_splits.append(q_bias2[row_start:row_end])

input2_splits= []
for i in range(4):
    col_start = i * c_split
    col_end = (i + 1) * c_split
    input2_splits.append(input2[:, col_start:col_end])

output2_splits = []
for i in range(4):
    row_start = i * r_split
    row_end = (i + 1) * r_split
    output2_splits.append(output2[:, row_start:row_end])

input2_encoded=[]
for j in range(4):
    input2_encoded.append(encode_vector(input2_splits[j][0].tolist(), 8))
input2_encoded_h=[]
for j in range(4):
    input2_encoded_h.append(bin_list_to_hex(input2_encoded[j], width_hex=2))
for i in range(4):
    
    with open(os.path.join(sim_dir, f'input_mem_{i}.mem'), 'w', encoding='utf-8') as f:
        f.write("\n".join(input2_encoded_h[i]) + "\n")

for i in range(4):
    val_list, col_idx_list, row_num_list, nnzs= csr_gen(q_w2_splits[i])
    print(nnzs)
    qw_data_mem, row_num_mem= encode_csr(val_list, col_idx_list, row_num_list)
    q_bias_mem_encoded= encode_vector(q_bias2_splits[i].tolist(), 32)

    qw_data_mem_h = bin_list_to_hex(qw_data_mem, width_hex=4)  # 15bit → 4 hex
    row_num_mem_h = bin_list_to_hex(row_num_mem, width_hex=2)
    q_bias_mem_h  = bin_list_to_hex(q_bias_mem_encoded,  width_hex=8)

    # qw_data_mem (len=933, 15-bit strings)
    with open(os.path.join(sim_dir, f'qw_data_mem_{i}.mem'), 'w', encoding='utf-8') as f:
        # f.write("\n".join(qw_data_mem) + "\n")
        f.write("\n".join(qw_data_mem_h) + "\n")

    # row_num_mem (len=128, 7-bit strings)
    with open(os.path.join(sim_dir, f'row_num_mem_{i}.mem'), 'w', encoding='utf-8') as f:
        # f.write("\n".join(row_num_mem) + "\n")
        f.write("\n".join(row_num_mem_h) + "\n")

    # q_bias_mem (len=128, 32-bit strings)
    with open(os.path.join(sim_dir, f'q_bias_mem_{i}.mem'), 'w', encoding='utf-8') as f:
        # f.write("\n".join(q_bias_mem_encoded) + "\n")
        f.write("\n".join(q_bias_mem_h) + "\n")

Z_out:-128.0, Scale_factor: 52.0
[887, 845, 806, 707]
[907, 853, 833, 647]
[898, 844, 794, 724]
[926, 864, 870, 701]


In [133]:
# i=0

# print(input2_splits[i][0][:])
# print(input2_encoded[i][:10])
# print(input2_encoded_h[i][:10])

PE=0
i=0
print(input2_splits[PE][0][i*10:(i+1)*10])
print(input2_splits[PE][0][(i+1)*10:(i+2)*10])
print(input2_splits[PE][0][(i+2)*10:(i+3)*10])
print(input2_splits[PE][0][(i+3)*10:(i+4)*10])

tensor([ -70., -128.,   25., -128., -128.,  -35.,  -18., -128., -128., -128.],
       grad_fn=<SliceBackward0>)
tensor([-127., -128., -128., -128.,  -32., -128., -128., -128., -128., -128.],
       grad_fn=<SliceBackward0>)
tensor([-128., -126., -128., -128., -128., -128.,  -79., -109., -128.,  -21.],
       grad_fn=<SliceBackward0>)
tensor([-117.,  -21., -126., -128., -128., -128.,  -12.,   15.,  -51., -107.],
       grad_fn=<SliceBackward0>)


In [134]:
#for tb_ (128x512) PE0

output_tmp2_00 = input2_splits[0].to(DEVICE) @ q_w2_splits[0][0].t().to(DEVICE)
output_tmp2_01 = input2_splits[1].to(DEVICE) @ q_w2_splits[0][1].t().to(DEVICE) + output_tmp2_00[0].to(DEVICE)
output_tmp2_02 = input2_splits[2].to(DEVICE) @ q_w2_splits[0][2].t().to(DEVICE) + output_tmp2_01[0].to(DEVICE)
output_tmp2_03 = input2_splits[3].to(DEVICE) @ q_w2_splits[0][3].t().to(DEVICE) + output_tmp2_02[0].to(DEVICE)
output_tmp2_fin1= output_tmp2_03 + q_bias2_splits[0].to(DEVICE)
output_tmp2_fin2=nn.ReLU()(output_tmp2_fin1)
output_tmp2_fin3= torch.floor(output_tmp2_fin2 * scale_factor2*(2**(-16))) + Z_out2
output_tmp2_fin4 = torch.clamp(output_tmp2_fin3, q_min_, q_max_).to(torch.int8)

output_tmp2={}
output_tmp2['PE0']=[]
output_tmp2['PE0'].append(output_tmp2_00)
output_tmp2['PE0'].append(output_tmp2_01)
output_tmp2['PE0'].append(output_tmp2_02)
output_tmp2['PE0'].append(output_tmp2_03)
output_tmp2['PE0'].append(output_tmp2_fin1)
output_tmp2['PE0'].append(output_tmp2_fin2)
output_tmp2['PE0'].append(output_tmp2_fin3)
output_tmp2['PE0'].append(output_tmp2_fin4)

#for tb_ (128x512) PE1
output_tmp2_11 = input2_splits[1].to(DEVICE) @ q_w2_splits[1][1].t().to(DEVICE)
output_tmp2_12 = input2_splits[2].to(DEVICE) @ q_w2_splits[1][2].t().to(DEVICE) + output_tmp2_11[0].to(DEVICE)
output_tmp2_13 = input2_splits[3].to(DEVICE) @ q_w2_splits[1][3].t().to(DEVICE) + output_tmp2_12[0].to(DEVICE)
output_tmp2_10 = input2_splits[0].to(DEVICE) @ q_w2_splits[1][0].t().to(DEVICE) + output_tmp2_13[0].to(DEVICE)
output_tmp2_fin1= output_tmp2_10 + q_bias2_splits[1].to(DEVICE)
output_tmp2_fin2=nn.ReLU()(output_tmp2_fin1)
output_tmp2_fin3= torch.floor(output_tmp2_fin2 * scale_factor2*(2**(-16))) + Z_out2
output_tmp2_fin4 = torch.clamp(output_tmp2_fin3, q_min_, q_max_).to(torch.int8)
output_tmp2['PE1']=[]
output_tmp2['PE1'].append(output_tmp2_11)
output_tmp2['PE1'].append(output_tmp2_12)
output_tmp2['PE1'].append(output_tmp2_13)
output_tmp2['PE1'].append(output_tmp2_10)
output_tmp2['PE1'].append(output_tmp2_fin1)
output_tmp2['PE1'].append(output_tmp2_fin2)
output_tmp2['PE1'].append(output_tmp2_fin3)
output_tmp2['PE1'].append(output_tmp2_fin4)

#for tb_ (128x512) PE2
output_tmp2_22 = input2_splits[2].to(DEVICE) @ q_w2_splits[2][2].t().to(DEVICE)
output_tmp2_23 = input2_splits[3].to(DEVICE) @ q_w2_splits[2][3].t().to(DEVICE) + output_tmp2_22[0].to(DEVICE)
output_tmp2_20 = input2_splits[0].to(DEVICE) @ q_w2_splits[2][0].t().to(DEVICE) + output_tmp2_23[0].to(DEVICE)
output_tmp2_21 = input2_splits[1].to(DEVICE) @ q_w2_splits[2][1].t().to(DEVICE) + output_tmp2_20[0].to(DEVICE)
output_tmp2_fin1= output_tmp2_21 + q_bias2_splits[2].to(DEVICE)
output_tmp2_fin2=nn.ReLU()(output_tmp2_fin1)
output_tmp2_fin3= torch.floor(output_tmp2_fin2 * scale_factor2*(2**(-16))) + Z_out2
output_tmp2_fin4 = torch.clamp(output_tmp2_fin3, q_min_, q_max_).to(torch.int8)
output_tmp2['PE2']=[]
output_tmp2['PE2'].append(output_tmp2_22)
output_tmp2['PE2'].append(output_tmp2_23)
output_tmp2['PE2'].append(output_tmp2_20)
output_tmp2['PE2'].append(output_tmp2_21)
output_tmp2['PE2'].append(output_tmp2_fin1)
output_tmp2['PE2'].append(output_tmp2_fin2)
output_tmp2['PE2'].append(output_tmp2_fin3)
output_tmp2['PE2'].append(output_tmp2_fin4)

#for tb_ (128x512) PE3
output_tmp2_33 = input2_splits[3].to(DEVICE) @ q_w2_splits[3][3].t().to(DEVICE)
output_tmp2_30 = input2_splits[0].to(DEVICE) @ q_w2_splits[3][0].t().to(DEVICE) + output_tmp2_33[0].to(DEVICE)
output_tmp2_31 = input2_splits[1].to(DEVICE) @ q_w2_splits[3][1].t().to(DEVICE) + output_tmp2_30[0].to(DEVICE)
output_tmp2_32 = input2_splits[2].to(DEVICE) @ q_w2_splits[3][2].t().to(DEVICE) + output_tmp2_31[0].to(DEVICE)
output_tmp2_fin1= output_tmp2_32 + q_bias2_splits[3].to(DEVICE)
output_tmp2_fin2=nn.ReLU()(output_tmp2_fin1)
output_tmp2_fin3= torch.floor(output_tmp2_fin2 * scale_factor2*(2**(-16))) + Z_out2
output_tmp2_fin4 = torch.clamp(output_tmp2_fin3, q_min_, q_max_).to(torch.int8)
output_tmp2['PE3']=[]
output_tmp2['PE3'].append(output_tmp2_33)
output_tmp2['PE3'].append(output_tmp2_30)
output_tmp2['PE3'].append(output_tmp2_31)
output_tmp2['PE3'].append(output_tmp2_32)
output_tmp2['PE3'].append(output_tmp2_fin1)
output_tmp2['PE3'].append(output_tmp2_fin2)
output_tmp2['PE3'].append(output_tmp2_fin3)
output_tmp2['PE3'].append(output_tmp2_fin4)

In [137]:
PE='PE0'
q_cnt=0
i=0
print(output_tmp2[PE][q_cnt][0][i*10:(i+1)*10])

tensor([-12732., -27794.,   1328., -23602., -12670., -23125.,  -2322., -16497.,
        -46229.,      0.], grad_fn=<SliceBackward0>)
